# Stage36 — GVZ·OVX 자산별 위험 오버레이

## 데이터 ZIP 하나로 재현하는 Google Colab 실행본

이 노트북은 로컬 `strategies` 패키지를 import하지 않습니다. Stage36과 실제 실행 경로에 필요한 Stage35·20·13·14·07·34·30·core 로직을 기능별 코드 셀로 풀어 넣었습니다.

### 사용 순서

1. 이 노트북을 Google Colab에서 엽니다.
2. **런타임 → 모두 실행**을 누릅니다.
3. 업로드 창이 뜨면 제공된 `stage36_colab_data.zip` 하나를 올립니다.
4. 약 수 분 뒤 2007-04~2026-07 성과표·위험회귀·bootstrap·월별 비중이 생성됩니다.

### 설계 경계

- GVZ는 GLD, OVX는 USO의 공분산 축만 조정합니다.
- GVZ/OVX로 기대수익 μ를 조정하지 않습니다.
- long-only, no leverage, 단일자산 과반금지 없음
- 출시 전 backfill·grid search·SJM/CJM·로지스틱 학습 없음
- 결과는 연구용 역사적 시뮬레이션이며 미래성과를 보장하지 않습니다.


## 원래 프로젝트 소스와 이 노트북 셀의 대응

| 원래 소스 | 이 노트북의 역할 |
|---|---|
| `stage36/.../asset_implied_volatility_risk_slsqp.py` | 10~16절: GVZ/OVX·DΣD·실험 |
| `stage35/.../earnings_credit_fundamentals_slsqp.py` | 9·11절: EPS·밸류·신용·기준 μ |
| `stage20/.../daily_technical_confidence_slsqp.py` | 8절: K-ratio·RSI·ATR |
| `stage13/.../economic_conditional_slsqp.py` | 3·6·7절: midrank·stress·조건부 μ/Σ |
| `stage14/.../dynamic_risk_slsqp.py` | 11~13절: bounds·비용·성과지표 |
| `stage07/.../zero_tune_strategy.py` | 4·5절: 원화 수익·거시확률·비용률 |
| `stage34/.../futures_basis_oi_confirmation_slsqp.py` | 14절: 미래위험 목적변수 |
| `stage30/.../abnormal_surface_erp_slsqp.py` | 13절: paired block bootstrap |
| `core/regime_research.py` | 4·11·13절: 자산목록·수익·CDaR |

노트북에는 이 경로에서 **실제로 호출되는 함수만** 포함합니다. 이전 Stage의 보고서·플롯 함수나 미사용 실험모델은 포함하지 않습니다.


## 먼저 잡아야 할 핵심: Stage35가 본체이고 Stage36은 위험센서다

Stage36 코드부터 읽으면 `μ35`와 `Σ35`가 갑자기 만들어지는 것처럼 보일 수 있습니다. 실제 계보는 다음과 같습니다.

> 거시경제가 **어떤 날씨인지** 판단하고 → VKOSPI·VIX6가 **폭풍이 얼마나 센지** 보고 → 가격·거래량이 **그 전망에 동의하는지** 확인하고 → 기업이익·밸류에이션이 **주식이 살 만한지** 검사하고 → 신용시장이 **금융시스템에 금이 가는지** 확인합니다. 이 정보가 Stage35의 기대수익 `μ35`와 위험행렬 `Σ35`로 압축됩니다. Stage36은 `μ35`는 그대로 두고 `Σ35`의 금·원유 축에 GVZ·OVX만 추가합니다.

### 전체 입력변수 지도

| 계층 | 원 입력 | 인과적 가공 | 최종 역할 |
|---|---|---|---|
| 성장 | GDP YoY, 수출 YoY, BSI | 과거만 이용한 percentile 후 평균 | 성장축 $g_t$ |
| 물가 | CPI YoY, PPI YoY, 수입물가 YoY | 과거만 이용한 percentile 후 평균 | 물가축 $\pi_t$ |
| 거시국면 | $g_t,\pi_t$ | 네 개 soft-regime 확률 | 기본 `μ`, 기본 `Σ` |
| 시장 공포 | VKOSPI 수준, 5일 로그변화 | expanding midrank | stress 수준·충격 |
| 옵션 표면 | VIX6 parallel shift, put/call skew, downside/upside convexity | 꼬리 비대칭·21일 지속성 | stress/recovery 보정 |
| 가격 추세 | 126일 K-ratio | $K/(1+\lvert K\rvert)$ | 거시 `μ`의 신뢰도 |
| 주식 확인 | 14일 price RSI, volume RSI | $(RSI-50)/50$ | KODEX200 방향 확인 |
| 실현 위험 | 14일 ATR/가격 | causal rank 후 $1+rank$ | 각 자산 `Σ` 축 확대 |
| 기업이익 | 12M forward EPS, 1M revision | 60개월 causal z-score·expanding slope | KODEX200 `μ` |
| 밸류에이션 | 12M forward PER, 국고채 10Y | $1/PER-y_{10Y}$ | KODEX200 장기 `μ` |
| 신용 | AA- 회사채 3Y−국고채 3Y, 20일 변화 | 60개월 rank·z-score | 주식 stress와 `Σ` |
| Stage36 | GVZ, OVX | 직전 월말 값의 252일 causal rank | 각각 GLD·USO `Σ` 축 |
| 최적화 상태 | 과거 4자산 월수익, 직전 보유비중 | downside·CDaR·drift | 위험제약·거래비용·초기점 |

### 코드가 최종적으로 만드는 두 장의 성적표

- `μ35`: 다음 한 달에 각 자산이 얼마나 유리할지 나타내는 월 기대수익 벡터입니다. 거시 조건부 수익을 기술신호로 덜 또는 더 신뢰하고, VKOSPI/VIX6 stress·recovery와 KODEX200의 EPS·밸류에이션을 반영합니다.
- `Σ35`: 각 자산의 흔들림과 동행관계를 나타내는 공분산 행렬입니다. 거시 조건부 공분산을 표본수에 따라 수축하고, ATR과 신용위험으로 해당 축을 확대합니다.

Stage36의 경계는 명확합니다.

\[
\mu_{36}=\mu_{35},\qquad
\Sigma_{36}=D_{GVZ,OVX}\Sigma_{35}D_{GVZ,OVX}
\]

즉 GVZ가 높다고 금의 기대수익을 깎거나, OVX가 높다고 원유를 매도하라는 방향 신호를 만들지 않습니다. 옵션시장이 비싸게 평가한 **미래 위험**만 포트폴리오 위험계산에 반영합니다.

### 이 노트북을 읽는 순서

1. 3~7절에서 인과적 변환 → 거시확률 → VKOSPI/VIX6 → 조건부 `μ·Σ`를 봅니다.
2. 8절에서 K-ratio·RSI가 `μ`를 뒤집지 않고 신뢰도만 조절하는 부분을 봅니다.
3. 9절에서 EPS·밸류에이션·신용이 KODEX200에 연결되는 정확한 식을 봅니다.
4. 10~11절에서 Stage36의 GVZ/OVX `DΣD`와 SLSQP 조립 순서를 봅니다.
5. 12~18절에서 월별 실행·성과·미래위험 진단·감사 결과를 확인합니다.

이 구현은 HMM·SJM·CJM·로지스틱 회귀로 국면을 학습하지 않습니다. 거시 여섯 변수로 네 확률을 직접 계산하며, 하이퍼파라미터 탐색으로 가장 좋아 보이는 문턱을 고르지도 않습니다.

In [ ]:
# Colab 기본환경에 없는 경우만 설치됩니다.
%pip -q install openpyxl statsmodels


## 1. 실행환경과 고정 설계값

이 셀은 Stage36 전체에서 공유하는 자산 순서와 경제적·수치적 상수를 선언합니다.

- 자산: `KODEX200`, `BOND`, `GLD`, `USO`
- 무레버리지·공매도 금지: 각 비중 0~1, 합계 1
- 위험 guard: 연환산 변동성 13%, 과거 CDaR(90%) -16%
- 거래비용: 전체 비중변화 15bp, 해외 순비중 변화 추가 5bp
- GVZ/OVX 최소 이력: 현재 값을 제외한 252개 일간관측
- SLSQP의 300회와 `ftol=1e-9`는 경제적 파라미터가 아니라 수치해석 설정입니다.

`configure_data_root`는 압축을 푼 데이터 폴더만 가리키며, 로컬 `strategies` 패키지를 참조하지 않습니다. `validate_data_bundle`은 ZIP에 기록된 파일 크기와 SHA-256을 전부 확인합니다.

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import shutil
import sqlite3
import unicodedata
import zipfile
from bisect import bisect_left, bisect_right, insort
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.optimize import minimize
from scipy.stats import spearmanr


ASSETS = ["KODEX200", "BOND", "GLD", "USO"]
REGIME_COLUMNS = [
    "p_Goldilocks",
    "p_Overheating",
    "p_Slowdown",
    "p_Stagflation",
]
FULL_START = pd.Period("2007-04", freq="M")
COMMON_START = pd.Period("2010-01", freq="M")
LOCKED_START = pd.Period("2018-01", freq="M")
RESEARCH_END = pd.Period("2026-07", freq="M")
ONE_WEEK = 5
ONE_TRADING_MONTH = 21
ONE_CALENDAR_YEAR = 12
MIN_CAUSAL_MONTHS = 60
MIN_SENSOR_HISTORY = 252
K_RATIO_DAYS = 126
WILDER_DAYS = 14
CREDIT_CHANGE_DAYS = 20
CATASTROPHE_ANNUAL_VOLATILITY = 0.13
CATASTROPHE_CDAR = 0.16
CDAR_CONFIDENCE = 0.90
DOMESTIC_TRADE_COST = 0.0015
FOREIGN_WEIGHT_CHANGE_COST = 0.0005
SLSQP_MAX_ITERATIONS = 300
SLSQP_TOLERANCE = 1e-9
NUMERICAL_EPSILON = 1e-12
UNCONSTRAINED_LONG_ONLY_BOUNDS = [(0.0, 1.0)] * len(ASSETS)
EQUITY_INDEX = ASSETS.index("KODEX200")
GOLD_INDEX = ASSETS.index("GLD")
OIL_INDEX = ASSETS.index("USO")
WEIGHT_COLUMNS = [f"w_{asset}" for asset in ASSETS]

DATA_ROOT: Path | None = None
RAW_DIR: Path | None = None
CACHE_DIR: Path | None = None
RESULTS_DIR: Path | None = None
OUTPUT_DIR: Path | None = None


def configure_data_root(data_root: str | Path) -> Path:
    """Point every loader at the extracted data-only bundle."""

    global DATA_ROOT, RAW_DIR, CACHE_DIR, RESULTS_DIR, OUTPUT_DIR
    DATA_ROOT = Path(data_root).resolve()
    RAW_DIR = DATA_ROOT / "raw_data"
    CACHE_DIR = DATA_ROOT / "cache"
    RESULTS_DIR = DATA_ROOT / "results"
    OUTPUT_DIR = DATA_ROOT / "colab_outputs"
    for path in (RAW_DIR, CACHE_DIR, RESULTS_DIR):
        if not path.is_dir():
            raise FileNotFoundError(f"Required data directory is missing: {path}")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    return DATA_ROOT


def sha256(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def get_path(directory: Path, filename: str) -> Path:
    """Resolve Korean filenames regardless of NFC/NFD normalization."""

    target = unicodedata.normalize("NFC", filename)
    for path in directory.iterdir():
        if unicodedata.normalize("NFC", path.name) == target:
            return path
    raise FileNotFoundError(filename)


def validate_data_bundle(data_root: str | Path) -> dict[str, Any]:
    root = Path(data_root)
    manifest_path = root / "manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    failures: list[str] = []
    for record in manifest["files"]:
        path = root / record["path"]
        if not path.is_file():
            failures.append(f"missing: {record['path']}")
        elif path.stat().st_size != record["bytes"]:
            failures.append(f"size: {record['path']}")
        elif sha256(path) != record["sha256"]:
            failures.append(f"sha256: {record['path']}")
    if failures:
        raise ValueError("Bundle validation failed: " + ", ".join(failures))
    return {
        "bundle": manifest["bundle"],
        "files": len(manifest["files"]),
        "bytes": int(sum(row["bytes"] for row in manifest["files"])),
        "all_hashes_match": True,
        "code_included": bool(manifest["code_included"]),
    }


## 2. 데이터 ZIP 하나 업로드하고 검증

Colab에서는 실행 시 표시되는 업로드 창에 `stage36_colab_data.zip` 하나만 올리면 됩니다. ZIP에는 코드가 없고 시장·거시·펀더멘털·GVZ/OVX 원천과 고정된 VIX6 6요인 일간 결과만 있습니다.

로컬 QA에서는 `STAGE36_DATA_ZIP` 환경변수를 사용합니다. 압축을 푼 뒤 16개 파일의 SHA-256이 manifest와 다르면 즉시 중단하므로, 깨진 업로드나 다른 버전의 데이터를 조용히 사용하는 일을 막습니다.

In [ ]:
def locate_or_upload_bundle() -> Path:
    """Use one uploaded ZIP in Colab; use an environment path in local QA."""

    local_override = os.environ.get("STAGE36_DATA_ZIP")
    if local_override:
        path = Path(local_override).resolve()
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    try:
        from google.colab import files  # type: ignore
    except ImportError as error:
        raise RuntimeError(
            "Local execution requires STAGE36_DATA_ZIP to point to "
            "stage36_colab_data.zip"
        ) from error
    print("stage36_colab_data.zip 파일 하나를 업로드하세요.")
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise ValueError("ZIP 파일을 정확히 하나만 업로드해야 합니다.")
    path = Path(zip_names[0]).resolve()
    path.write_bytes(uploaded[zip_names[0]])
    return path


def extract_bundle(bundle_zip: str | Path) -> Path:
    """Extract to a clean work directory and validate all source hashes."""

    default_work = "/content/stage36_workspace"
    work_root = Path(os.environ.get("STAGE36_WORK_DIR", default_work)).resolve()
    if work_root.exists():
        shutil.rmtree(work_root)
    work_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(bundle_zip) as archive:
        archive.extractall(work_root)
    data_root = work_root / "stage36_data"
    configure_data_root(data_root)
    audit = validate_data_bundle(data_root)
    print(json.dumps(audit, ensure_ascii=False, indent=2))
    return data_root


In [ ]:
bundle_zip = locate_or_upload_bundle()
data_root = extract_bundle(bundle_zip)
print(f"사용 데이터 루트: {data_root}")


## 3. 인과적 변환: expanding percentile·midrank·z-score

현재 관측치는 현재까지 알려진 역사 안에서만 평가합니다. 미래 전체표본 평균이나 표준편차는 쓰지 않습니다.

\[
q_t=\frac{L_t+0.5(E_t+1)}{n_t+1}
\]

`causal_expanding_midrank`는 이 empirical CDF를 정렬목록으로 계산합니다. GVZ/OVX는 현재 값 이전의 유효관측이 252개 미만이면 순위가 있더라도 비활성화됩니다. 펀더멘털 z-score는 현재월을 제외한 최소 60개월의 평균과 표준편차를 사용합니다.

### 왜 이 변환이 필요한가

GDP 3%, BSI 95, GVZ 24처럼 단위가 다른 원값을 그대로 평균하면 숫자가 큰 변수가 결과를 지배합니다. percentile은 각 값을 “자기 역사에서 어느 위치인가”라는 공통 0~1 척도로 바꿉니다. 중요한 점은 전체 기간이 아니라 **그 시점까지 들어온 값만** 정렬목록에 넣는다는 것입니다.

예를 들어 현재 값보다 작은 과거 관측이 `L`, 같은 값까지 포함한 동률 수가 `E`, 현재를 포함한 관측수가 `n+1`이면 midrank를 사용합니다. 동일한 값이 반복돼도 먼저 나온 관측과 나중 관측을 자의적으로 다르게 취급하지 않기 위해서입니다.

`causal_zscore`도 같은 원칙을 따릅니다. 현재월 값은 현재월 이전 평균·표준편차와 비교합니다. 최소 60개월이 쌓이기 전에는 펀더멘털 신호를 활성화하지 않습니다. GVZ·OVX에는 더 촘촘한 일간자료를 쓰므로 현재일을 제외한 252개 유효관측을 요구합니다.

**코드에서 확인할 부분**

- `shift(1)`: 현재값이 자기 기준통계에 들어가는 것을 차단합니다.
- `prior_count >= 252`: 센서 출시 초기의 불안정한 순위를 중립 배수 1로 둡니다.
- expanding 방식: 2020년 값을 평가하면서 2025년 분포를 보는 전 표본 look-ahead를 막습니다.

In [ ]:
def causal_expanding_percentile(series: pd.Series) -> pd.Series:
    """Empirical midrank using observations available through the current row."""

    output = pd.Series(np.nan, index=series.index, dtype=float)
    history: list[float] = []
    for index, value in series.items():
        if not np.isfinite(value):
            continue
        history.append(float(value))
        reference = np.asarray(history, dtype=float)
        less = float(np.sum(reference < value))
        equal = float(np.sum(reference == value))
        output.loc[index] = (less + 0.5 * equal) / len(reference)
    return output


def causal_expanding_midrank(series: pd.Series) -> pd.Series:
    """O(n log n) causal empirical CDF with equal-value midranks."""

    result = pd.Series(np.nan, index=series.index, dtype=float)
    ordered: list[float] = []
    for index, raw_value in series.items():
        value = float(raw_value) if pd.notna(raw_value) else np.nan
        if not np.isfinite(value):
            continue
        left = bisect_left(ordered, value)
        right = bisect_right(ordered, value)
        equal_after_insertion = right - left + 1
        result.loc[index] = (
            left + 0.5 * equal_after_insertion
        ) / (len(ordered) + 1)
        insort(ordered, value)
    return result


def causal_zscore(series: pd.Series, minimum: int = MIN_CAUSAL_MONTHS) -> pd.Series:
    values = series.astype(float)
    prior = values.shift(1)
    mean = prior.expanding(min_periods=minimum).mean()
    std = prior.expanding(min_periods=minimum).std(ddof=1)
    return ((values - mean) / std.where(std > 0.0)).replace(
        [np.inf, -np.inf], np.nan
    )


def rank_after_prior_history(
    series: pd.Series, minimum: int = MIN_SENSOR_HISTORY
) -> tuple[pd.Series, pd.Series]:
    rank = causal_expanding_midrank(series)
    prior_count = (
        series.notna().shift(1).fillna(False).astype(int).cumsum().astype(int)
    )
    return rank.where(prior_count >= minimum), prior_count


## 4. 네 자산의 원화 월수익률

전략 수익률은 월 첫 거래일 시가에서 다음 달 첫 거래일 시가까지 계산합니다. KODEX200은 2009년 3월까지 `compass.db`의 KOSPI200 프록시를 실제 ETF에 레벨 정합해 연결합니다. BOND는 KRX 채권 총수익지수, GLD와 USO는 USD 가격에 USDKRW를 곱해 한국 투자자의 원화 수익률로 바꿉니다.

이 셀은 인터넷에서 가격을 다시 받지 않습니다. 따라서 Colab 실행일과 무관하게 동일한 데이터 스냅샷을 사용합니다.

### 왜 월 첫 시가에서 다음 달 첫 시가인가

전략은 직전 월말까지 알려진 신호를 이용해 다음 달 비중을 정합니다. 그러므로 신호를 관찰할 수 있는 시점과 체결 수익률의 시작점을 분리해야 합니다. 목표월 첫 거래일 시가에서 진입하고 다음 달 첫 거래일 시가까지 보유하는 정의는 “월말 정보를 본 뒤 다음 거래가능 시점에 체결한다”는 시간순서를 구현합니다.

GLD와 USO는 달러가격만 보면 한국 투자자의 실제 손익과 다릅니다. 코드가 일간 OHLC와 월수익률 모두에 USDKRW를 곱하는 이유는 환율 효과를 빠뜨리지 않고 기술신호와 백테스트 수익률의 통화기준도 일치시키기 위해서입니다.

KODEX200 ETF가 충분히 길지 않은 초기구간은 `compass.db`의 KOSPI200 프록시를 실제 ETF 첫 관측 레벨에 맞춰 연결합니다. 이 조정은 수익률 흐름을 이어 주기 위한 레벨 정합이며 미래 수익을 채우는 backfill과는 다릅니다.

In [ ]:
def load_market_cache() -> pd.DataFrame:
    assert CACHE_DIR is not None
    path = CACHE_DIR / "market_daily.csv"
    return pd.read_csv(path, parse_dates=["date"])


def load_monthly_asset_returns() -> tuple[pd.DataFrame, pd.DataFrame]:
    """Reproduce the exact monthly open-to-next-open KRW return panel."""

    assert RAW_DIR is not None
    market = load_market_cache()
    with sqlite3.connect(get_path(RAW_DIR, "compass.db")) as connection:
        proxy = pd.read_sql(
            "select date, open, close from etf_prices "
            "where symbol = ? order by date",
            connection,
            params=("1028",),
        )
    proxy["date"] = pd.to_datetime(proxy["date"])
    proxy[["open", "close"]] = proxy[["open", "close"]].apply(
        pd.to_numeric, errors="coerce"
    )

    actual = market.loc[market["symbol"].eq("KODEX200")].copy()
    actual = actual.dropna(subset=["open"])
    actual = actual.loc[actual["date"] > pd.Timestamp("2009-03-31")]
    first_actual = actual["date"].min()
    actual_anchor = float(
        actual.loc[actual["date"].eq(first_actual), "open"].iloc[0]
    )
    proxy_anchor = proxy.loc[proxy["date"].eq(first_actual), "open"]
    if proxy_anchor.empty:
        nearest = proxy.iloc[(proxy["date"] - first_actual).abs().argsort()[:1]]
        proxy_anchor_value = float(nearest["open"].iloc[0])
    else:
        proxy_anchor_value = float(proxy_anchor.iloc[0])
    for column in ("open", "close"):
        proxy[column] *= actual_anchor / proxy_anchor_value
    proxy = proxy.loc[proxy["date"] < first_actual].copy()
    proxy["symbol"] = "KODEX200"
    kodex = pd.concat(
        [proxy[["date", "symbol", "open", "close"]], actual],
        ignore_index=True,
    )

    bond = pd.read_csv(get_path(RAW_DIR, "krx_bond_index.csv"), encoding="cp949")
    bond["date"] = pd.to_datetime(bond.iloc[:, 0])
    bond["open"] = (
        bond.iloc[:, 1].astype(str).str.replace(",", "", regex=False).astype(float)
    )
    bond["close"] = bond["open"]
    bond["symbol"] = "BOND"

    fx = (
        market.loc[market["symbol"].eq("USDKRW")]
        .set_index("date")["close"]
        .sort_index()
    )
    fx = fx.reindex(pd.date_range(fx.index.min(), fx.index.max(), freq="D")).ffill()
    first_open: dict[str, pd.Series] = {}
    for symbol, data in {
        "KODEX200": kodex,
        "BOND": bond,
        "GLD": market.loc[market["symbol"].eq("GLD")],
        "USO": market.loc[market["symbol"].eq("USO")],
    }.items():
        temp = data.dropna(subset=["open"]).sort_values("date").copy()
        temp["month"] = temp["date"].dt.to_period("M")
        first = temp.groupby("month", sort=True).first()
        value = first["open"].astype(float)
        if symbol in {"GLD", "USO"}:
            aligned = fx.reindex(pd.DatetimeIndex(first["date"]), method="ffill")
            value = value * aligned.to_numpy()
        first_open[symbol] = value
    levels = pd.concat(first_open, axis=1).sort_index()
    returns = levels.shift(-1).div(levels).sub(1.0).dropna(how="any")
    return returns[ASSETS], levels[ASSETS]


## 5. 무학습 거시 국면 확률

GDP·수출·BSI의 인과적 백분위 평균을 성장확률 `g`, CPI·PPI·수입물가 평균을 물가확률 `π`로 둡니다.

\[
p_G=g(1-\pi),\quad p_O=g\pi,\quad
p_S=(1-g)(1-\pi),\quad p_{Stag}=(1-g)\pi
\]

하드 분류, 로지스틱 회귀, SJM/CJM 학습은 없습니다. 네 연속확률은 합이 1이며 과거 수익의 국면별 가중치가 됩니다. GDP와 수출에는 원 코드와 같은 한 달 공표 지연, 물가에는 두 달 공표 지연을 적용합니다.

### 경제 질문을 두 축으로 압축한다

성장축은 “경제 엔진이 얼마나 잘 도는가”를 묻습니다. GDP는 경제 전체, 수출은 한국경제의 대외수요, BSI는 기업 현장의 전망을 보완합니다. 세 percentile이 각각 0.8, 0.7, 0.6이라면

\[
g_t=(0.8+0.7+0.6)/3=0.7
\]

로 읽습니다. 물가축은 “엔진이 과열됐는가”를 묻습니다. CPI는 소비자단계, PPI는 생산자단계, 수입물가는 해외 원가압력을 포착합니다.

### 하드 라벨 대신 혼합상태를 쓰는 이유

예를 들어 $g=0.7$, $\pi=0.8$이면 네 확률은 다음과 같습니다.

| 국면 | 계산 | 확률 |
|---|---:|---:|
| Goldilocks | $0.7(1-0.8)$ | 14% |
| Overheating | $0.7\times0.8$ | 56% |
| Slowdown | $(1-0.7)(1-0.8)$ | 6% |
| Stagflation | $(1-0.7)0.8$ | 24% |

현실 경제는 한 달 사이에 골디락스에서 스태그플레이션으로 완전히 점프하지 않습니다. “과열 성격이 가장 강하지만 스태그플레이션 위험도 일부 있다”처럼 확률을 나누면 경계 부근의 작은 데이터변화가 포트폴리오를 통째로 뒤집는 일을 줄일 수 있습니다.

이 확률은 예측모델이 출력한 class probability가 아닙니다. 성장·물가 percentile의 곱으로 직접 계산되므로 학습표본 부족으로 로지스틱 계수가 흔들리는 문제도 없습니다.

In [ ]:
def load_macro_levels() -> pd.DataFrame:
    """Load six published macro levels with the original release lags."""

    assert RAW_DIR is not None
    gdp = pd.read_excel(
        get_path(RAW_DIR, "GDP 성장률.xlsx"), index_col=0, skiprows=6
    )
    gdp.columns = ["GDP_QoQ", "GDP_YoY"]
    gdp.index = (
        pd.PeriodIndex(gdp.index, freq="Q")
        .asfreq("M", how="end")
        .to_timestamp("M")
        + pd.offsets.MonthEnd(1)
    )
    gdp = gdp.resample("ME").ffill()

    trade = pd.read_excel(
        get_path(RAW_DIR, "수출입 총괄_20260816.xlsx"),
        index_col=0,
        skiprows=4,
    )
    trade = trade[["수출 금액", "수입금액"]].iloc[1:].copy()
    for column in trade:
        trade[column] = (
            trade[column].astype(str).str.replace(",", "", regex=False).astype(float)
        )
    trade.index = pd.to_datetime(trade.index, format="%Y.%m") + pd.offsets.MonthEnd(1)
    trade["Export_YoY"] = trade["수출 금액"].pct_change(12) * 100.0

    bsi = pd.read_csv(
        get_path(RAW_DIR, "기업경기조사(전망).csv"), encoding="cp949"
    )
    bsi = bsi.loc[
        bsi["업종코드별"].eq("제 조 업")
        & bsi["BSI코드별"].eq("업황전망BSI 1)")
    ].iloc[:, 2:4]
    bsi["시점"] = (
        bsi["시점"]
        .str.replace("월", "", regex=False)
        .str.replace(" ", "", regex=False)
    )
    bsi["시점"] = pd.to_datetime(bsi["시점"], format="%Y.%m") + pd.offsets.MonthEnd(1)
    bsi = bsi.set_index("시점")
    bsi.columns = ["BSI"]

    cpi = pd.read_excel(
        get_path(RAW_DIR, "소비자물가 상승률.xlsx"), index_col=0, skiprows=6
    )
    cpi.columns = ["CPI_QoQ", "CPI_YoY"]
    cpi.index = pd.to_datetime(cpi.index, format="%Y-%m") + pd.offsets.MonthEnd(2)

    ppi = pd.read_excel(
        get_path(RAW_DIR, "생산자물가 상승률.xlsx"), index_col=0, skiprows=6
    )
    ppi.columns = ["PPI_QoQ", "PPI_YoY"]
    ppi.index = pd.to_datetime(ppi.index, format="%Y-%m") + pd.offsets.MonthEnd(2)

    prices = pd.read_excel(
        get_path(RAW_DIR, "수출입물가 상승률.xlsx"), index_col=0, skiprows=6
    )
    prices.columns = ["ExportPrice_YoY", "ImportPrice_YoY"]
    prices.index = pd.to_datetime(prices.index, format="%Y-%m") + pd.offsets.MonthEnd(2)

    return pd.concat(
        [
            gdp["GDP_YoY"],
            trade["Export_YoY"],
            bsi["BSI"],
            cpi["CPI_YoY"],
            ppi["PPI_YoY"],
            prices["ImportPrice_YoY"],
        ],
        axis=1,
    ).sort_index()


def build_macro_probabilities(
    returns: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    levels = load_macro_levels()
    ranks = levels.apply(causal_expanding_percentile)
    rows: list[dict[str, Any]] = []
    for target_month in returns.index:
        signal_month = target_month - 1
        known = ranks.loc[: signal_month.to_timestamp("M")]
        if known.empty:
            continue
        current = known.iloc[-1]
        if current.isna().any():
            continue
        growth = float(current[["GDP_YoY", "Export_YoY", "BSI"]].mean())
        inflation = float(
            current[["CPI_YoY", "PPI_YoY", "ImportPrice_YoY"]].mean()
        )
        rows.append(
            {
                "target_month": target_month,
                "signal_month": signal_month,
                "p_growth_high": growth,
                "p_inflation_high": inflation,
            }
        )
    probabilities = pd.DataFrame(rows).set_index("target_month")
    probabilities.index = pd.PeriodIndex(probabilities.index, freq="M")
    growth = probabilities["p_growth_high"]
    inflation = probabilities["p_inflation_high"]
    probabilities["p_Goldilocks"] = growth * (1.0 - inflation)
    probabilities["p_Overheating"] = growth * inflation
    probabilities["p_Slowdown"] = (1.0 - growth) * (1.0 - inflation)
    probabilities["p_Stagflation"] = (1.0 - growth) * inflation
    return probabilities, ranks


## 6. VKOSPI·VIX6 연속 스트레스와 회복

네 블록을 동일가중으로 결합합니다.

1. VKOSPI 수준
2. VKOSPI 5일 변화와 VIX6 parallel shift
3. put/call skew 및 downside/upside convexity의 왼쪽 꼬리 비대칭
4. 앞선 세 블록의 21거래일 지속성

공포 상승은 즉시 반영하고, 하락 시에는 현재 stress와 5일 평균 중 큰 값을 써서 하루짜리 안도 랠리가 완전한 회복으로 해석되는 것을 줄입니다. 목표월에는 직전 월말까지의 마지막 일간값만 연결합니다.

### 거시데이터보다 빠른 시장의 공포를 읽는다

GDP가 양호해도 금융시장은 며칠 만에 무너질 수 있습니다. VKOSPI 수준은 공포가 구조적으로 높은지를, 5일 로그변화는 공포가 갑자기 폭발했는지를 구분합니다. VKOSPI 35가 한 달 지속된 경우와 18에서 35로 급등한 경우가 다른 이유입니다.

VIX6는 공포의 **높이뿐 아니라 모양**을 봅니다.

- `parallel_shift`: 옵션 전반의 내재변동성이 함께 이동했는지
- `put_skew`, `downside_convexity`: 하락보험과 극단 하락꼬리가 비싸졌는지
- `call_skew`, `upside_convexity`: 상승꼬리와 비교했을 때 하락꼬리 비대칭이 얼마나 큰지
- 21일 persistence: 하루짜리 충격인지 약 한 달 이어진 상태인지

`stress_score=max(stress_raw, 최근 5일 평균)`은 공포 급등은 바로 올리되, 하루 하락만으로 위험이 완전히 사라졌다고 보지 않는 간단한 히스테리시스입니다. `recovery_score`는 5일 평균보다 현재 stress가 낮아진 정도를 역사적 순위로 바꿉니다.

**시간축 확인:** 목표월 $t$에는 $t-1$ 월말까지 관찰된 마지막 일간값만 사용합니다. 이 셀의 미래 데이터는 월별 신호에 연결되지 않습니다.

In [ ]:
def load_vkospi_daily() -> pd.DataFrame:
    assert RAW_DIR is not None
    raw = pd.read_csv(RAW_DIR / "VKOSPIData.csv", encoding="utf-8-sig")
    daily = raw.iloc[:, :7].copy()
    daily.columns = ["date", "close", "change", "return_pct", "open", "high", "low"]
    daily["date"] = pd.to_datetime(daily["date"], format="%Y/%m/%d", errors="coerce")
    daily["close"] = pd.to_numeric(
        daily["close"].astype(str).str.replace(",", "", regex=False),
        errors="coerce",
    )
    daily = daily.dropna(subset=["date", "close"]).set_index("date").sort_index()
    return daily.loc[~daily.index.duplicated(keep="last")]


def read_vix6_components() -> pd.DataFrame:
    assert RESULTS_DIR is not None
    path = RESULTS_DIR / "vix6_case1_features_daily.csv"
    frame = pd.read_csv(path, index_col=0, parse_dates=True)
    components = [
        "sticky_strike",
        "parallel_shift",
        "put_skew",
        "call_skew",
        "downside_convexity",
        "upside_convexity",
    ]
    missing = [column for column in components if column not in frame]
    if missing:
        raise ValueError(f"VIX6 cache is missing columns: {missing}")
    return frame[components].apply(pd.to_numeric, errors="coerce").sort_index()


def build_daily_stress_features() -> pd.DataFrame:
    vkospi = load_vkospi_daily()[["close"]].rename(columns={"close": "vkospi_close"})
    daily = vkospi.join(read_vix6_components(), how="inner").sort_index()
    daily = daily.loc[~daily.index.duplicated(keep="last")]
    daily["vkospi_log_change_5"] = np.log(daily["vkospi_close"]).diff(ONE_WEEK)
    daily["vix6_left_impulse"] = daily["put_skew"] + daily["downside_convexity"]
    daily["vix6_right_impulse"] = daily["call_skew"] + daily["upside_convexity"]
    daily["vix6_tail_asymmetry"] = (
        daily["vix6_left_impulse"] - daily["vix6_right_impulse"]
    )
    daily["level_component"] = causal_expanding_midrank(daily["vkospi_close"])
    daily["vkospi_shock_rank"] = causal_expanding_midrank(daily["vkospi_log_change_5"])
    daily["parallel_shift_rank"] = causal_expanding_midrank(daily["parallel_shift"])
    daily["shock_component"] = daily[
        ["vkospi_shock_rank", "parallel_shift_rank"]
    ].mean(axis=1)
    daily["left_impulse_rank"] = causal_expanding_midrank(daily["vix6_left_impulse"])
    daily["tail_asymmetry_rank"] = causal_expanding_midrank(
        daily["vix6_tail_asymmetry"]
    )
    daily["tail_component"] = daily[
        ["left_impulse_rank", "tail_asymmetry_rank"]
    ].mean(axis=1)
    three_blocks = daily[
        ["level_component", "shock_component", "tail_component"]
    ].mean(axis=1)
    daily["persistence_component"] = three_blocks.rolling(
        ONE_TRADING_MONTH, min_periods=1
    ).mean()
    daily["stress_raw"] = daily[
        ["level_component", "shock_component", "tail_component", "persistence_component"]
    ].mean(axis=1)
    one_week_mean = daily["stress_raw"].rolling(ONE_WEEK, min_periods=1).mean()
    daily["stress_score"] = pd.concat(
        [daily["stress_raw"], one_week_mean], axis=1
    ).max(axis=1).clip(0.0, 1.0)
    recovery_intensity = (one_week_mean - daily["stress_raw"]).clip(lower=0.0)
    daily["recovery_score"] = causal_expanding_midrank(
        recovery_intensity.where(recovery_intensity > 0.0)
    ).fillna(0.0)
    return daily.replace([np.inf, -np.inf], np.nan)


def build_monthly_stress_signals(
    target_months: pd.PeriodIndex, daily: pd.DataFrame
) -> pd.DataFrame:
    columns = [
        "level_component",
        "shock_component",
        "tail_component",
        "persistence_component",
        "stress_raw",
        "stress_score",
        "recovery_score",
    ]
    valid = daily.dropna(subset=["stress_score"])
    rows: list[dict[str, Any]] = []
    for target_month in target_months:
        signal_month = target_month - 1
        known = valid.loc[: signal_month.to_timestamp("M")]
        if known.empty:
            continue
        current = known.iloc[-1]
        rows.append(
            {
                "target_month": target_month,
                "stress_signal_month": signal_month,
                "stress_signal_date": known.index[-1],
                **{column: float(current[column]) for column in columns},
            }
        )
    monthly = pd.DataFrame(rows).set_index("target_month")
    monthly.index = pd.PeriodIndex(monthly.index, freq="M")
    return monthly


## 7. soft-regime 조건부 평균과 공분산

과거 각 월의 네 국면확률을 가중치로 사용해 국면별 평균과 공분산을 추정합니다. 유효표본이 작으면 무조건부 모멘트로 수축합니다.

\[
n_{eff}=\frac{(\sum p_i)^2}{\sum p_i^2},\qquad
c=\frac{n_{eff}}{n_{eff}+12}
\]

VKOSPI/VIX6 stress와 recovery의 국면별 OLS 기울기는 자기 R²로 신뢰도를 낮춥니다. 주식과 원유는 스트레스 기울기≤0, 회복 기울기≥0이라는 고정 경제부호를 둡니다. 음의 고유값은 수치오차 수준의 바닥만 적용해 PSD 행렬로 만듭니다.

### 네 국면확률이 기본 `μ`와 `Σ`가 되는 과정

과거 월 $i$가 Goldilocks 확률 70%였다면 그 월 수익은 Goldilocks 통계에 0.7만큼 기여합니다. 이렇게 각 국면의 가중평균수익 $\tilde\mu_k$와 가중공분산 $\tilde\Sigma_k$를 계산합니다. 질문을 사람말로 바꾸면 “과거에 이 국면의 냄새가 강했던 달에 네 자산이 평균적으로 어떻게 움직였나?”입니다.

### 표본이 적으면 전체시장 통계로 수축한다

확률가중치가 몇 달에 몰리면 실제 정보량은 달력상의 월수보다 작습니다.

\[
n_{eff,k}=\frac{(\sum_i p_{i,k})^2}{\sum_i p_{i,k}^2},\qquad
c_k=\frac{n_{eff,k}}{n_{eff,k}+12}
\]

따라서

\[
\mu_k=c_k\tilde\mu_k+(1-c_k)\mu_{all},\qquad
\Sigma_k=c_k\tilde\Sigma_k+(1-c_k)\Sigma_{all}
\]

이 됩니다. 리뷰 2개의 평점 5.0을 리뷰 5,000개의 평점 4.8만큼 믿지 않는 것과 같습니다. `12`는 1년의 무조건부 prior를 뜻하며 후보탐색으로 고른 값이 아닙니다.

현재 거시확률로 네 국면 통계를 다시 섞어 $\mu_{macro,t}=\sum_k p_{t,k}\mu_k$를 만듭니다. stress/recovery 회귀기울기는 자기 $R^2$에 해당하는 reliability로 축소합니다. 주식·원유에는 stress 계수≤0, recovery 계수≥0이라는 사전 경제부호를 두어 표본우연이 “공포 급등은 주식에 호재” 같은 역방향 규칙을 만드는 것을 막습니다.

공분산도 현재 stress가 높을수록 국면별 high-stress 공분산 쪽으로 이동합니다. 마지막 `nearest_psd`는 음의 고유값 때문에 $w'\Sigma w$가 음수가 되는 수치오류를 차단합니다.

In [ ]:
def nearest_psd(covariance: np.ndarray) -> np.ndarray:
    symmetric = 0.5 * (covariance + covariance.T)
    eigenvalues, eigenvectors = np.linalg.eigh(symmetric)
    scale = max(float(np.trace(symmetric)) / len(symmetric), NUMERICAL_EPSILON)
    floor = scale * 1e-10
    return (eigenvectors * np.maximum(eigenvalues, floor)) @ eigenvectors.T


def weighted_mean_and_covariance(
    values: np.ndarray, weights: np.ndarray
) -> tuple[np.ndarray, np.ndarray]:
    weights = np.asarray(weights, dtype=float)
    values = np.asarray(values, dtype=float)
    total = float(weights.sum())
    if total <= NUMERICAL_EPSILON:
        weights = np.ones(len(values), dtype=float)
        total = float(len(values))
    normalized = weights / total
    mean = normalized @ values
    centered = values - mean
    covariance = (centered * normalized[:, None]).T @ centered
    return mean, nearest_psd(covariance)


def estimate_conditional_moments(
    history: pd.DataFrame,
    historical_probabilities: pd.DataFrame,
    current_probabilities: pd.Series,
    historical_stress: pd.Series,
    current_stress: float,
    historical_recovery: pd.Series,
    current_recovery: float,
) -> tuple[np.ndarray, np.ndarray, dict[str, Any]]:
    common = history.index.intersection(historical_probabilities.index)
    common = common.intersection(historical_stress.dropna().index)
    common = common.intersection(historical_recovery.dropna().index)
    if len(common) < ONE_CALENDAR_YEAR:
        raise ValueError("At least one calendar year of causal history is required.")
    values = history.loc[common, ASSETS].to_numpy(dtype=float)
    probabilities = historical_probabilities.loc[common, REGIME_COLUMNS]
    stress = historical_stress.loc[common].to_numpy(dtype=float)
    recovery = historical_recovery.loc[common].to_numpy(dtype=float)
    current_p = current_probabilities[REGIME_COLUMNS].to_numpy(dtype=float)
    current_p = current_p / current_p.sum()
    current_s = float(np.clip(current_stress, 0.0, 1.0))
    current_r = float(np.clip(current_recovery, 0.0, 1.0))
    macro_mean = np.zeros(len(ASSETS), dtype=float)
    macro_covariance = np.zeros((len(ASSETS), len(ASSETS)), dtype=float)
    high_stress_covariance = np.zeros_like(macro_covariance)
    stress_adjustment = np.zeros(len(ASSETS), dtype=float)
    unconditional_mean, unconditional_covariance = weighted_mean_and_covariance(
        values, np.ones(len(values), dtype=float)
    )
    effective_samples: dict[str, float] = {}
    credibility_rows: dict[str, float] = {}
    for regime_index, regime_column in enumerate(REGIME_COLUMNS):
        regime_weights = probabilities[regime_column].to_numpy(dtype=float)
        raw_mean, raw_covariance = weighted_mean_and_covariance(values, regime_weights)
        effective_sample = float(
            regime_weights.sum() ** 2
            / max(float(np.square(regime_weights).sum()), NUMERICAL_EPSILON)
        )
        credibility = effective_sample / (effective_sample + ONE_CALENDAR_YEAR)
        regime_mean = credibility * raw_mean + (1.0 - credibility) * unconditional_mean
        regime_covariance = (
            credibility * raw_covariance
            + (1.0 - credibility) * unconditional_covariance
        )
        weighted_asset_variance = (
            regime_weights[:, None] * (values - raw_mean) ** 2
        ).sum(axis=0)

        def reliable_slope(feature: np.ndarray) -> tuple[np.ndarray, float]:
            feature_mean = float(np.average(feature, weights=regime_weights))
            centered = feature - feature_mean
            denominator = float(np.sum(regime_weights * centered**2))
            if denominator <= NUMERICAL_EPSILON:
                return np.zeros(len(ASSETS), dtype=float), feature_mean
            raw_slope = (
                (regime_weights * centered)[:, None] * (values - raw_mean)
            ).sum(axis=0) / denominator
            covariance_numerator = raw_slope * denominator
            reliability = np.divide(
                covariance_numerator**2,
                denominator * weighted_asset_variance,
                out=np.zeros(len(ASSETS), dtype=float),
                where=weighted_asset_variance > NUMERICAL_EPSILON,
            ).clip(0.0, 1.0)
            return raw_slope * reliability, feature_mean

        beta, stress_mean = reliable_slope(stress)
        recovery_beta, recovery_mean = reliable_slope(recovery)
        beta[EQUITY_INDEX] = min(beta[EQUITY_INDEX], 0.0)
        beta[OIL_INDEX] = min(beta[OIL_INDEX], 0.0)
        recovery_beta[EQUITY_INDEX] = max(recovery_beta[EQUITY_INDEX], 0.0)
        recovery_beta[OIL_INDEX] = max(recovery_beta[OIL_INDEX], 0.0)
        stress_weights = regime_weights * stress
        _, raw_stress_covariance = weighted_mean_and_covariance(values, stress_weights)
        stress_effective = float(
            stress_weights.sum() ** 2
            / max(float(np.square(stress_weights).sum()), NUMERICAL_EPSILON)
        )
        stress_credibility = stress_effective / (stress_effective + ONE_CALENDAR_YEAR)
        regime_stress_covariance = (
            stress_credibility * raw_stress_covariance
            + (1.0 - stress_credibility) * regime_covariance
        )
        probability = float(current_p[regime_index])
        macro_mean += probability * regime_mean
        macro_covariance += probability * regime_covariance
        high_stress_covariance += probability * regime_stress_covariance
        stress_adjustment += probability * (
            beta * (current_s - stress_mean)
            + recovery_beta * (current_r - recovery_mean)
        )
        effective_samples[regime_column] = effective_sample
        credibility_rows[regime_column] = credibility
    covariance = (
        (1.0 - current_s) * macro_covariance
        + current_s * high_stress_covariance
    )
    detail = {
        "macro_expected_monthly_return": macro_mean.tolist(),
        "stress_return_adjustment": stress_adjustment.tolist(),
        "effective_regime_samples": effective_samples,
        "regime_credibility": credibility_rows,
    }
    return macro_mean + stress_adjustment, nearest_psd(covariance), detail


## 8. K-ratio·RSI·ATR 기술 입력

`K_RATIO_DAYS=126`은 약 6개월 로그가격 추세의 기울기 안정성을, Wilder 14일 RSI와 ATR은 방향 강도와 현재 위험을 측정합니다.

- 모든 자산: K-ratio와 ATR/NATR의 인과적 백분위
- KODEX200: price RSI와 거래량 RSI를 추가
- 기술 방향과 거시 상대방향이 일치할수록 거시 기대수익의 횡단면 차이를 더 신뢰
- ATR 순위는 각 자산 공분산 축을 `1+rank`로 확대

해외자산 OHLC도 USDKRW로 원화 환산하므로 월수익률과 기술지표의 통화기준이 같습니다.

### K-ratio: 추세의 방향뿐 아니라 안정성을 본다

126일 로그가격에 직선을 적합하고 기울기를 그 기울기의 표준오차로 나눈 뒤 기간규모를 조정합니다. 같은 상승률이라도 들쭉날쭉한 가격보다 꾸준한 상승경로의 K-ratio가 큽니다. 극단값이 전략을 장악하지 않도록

\[
Kscore=\frac{K}{1+|K|},\qquad -1<Kscore<1
\]

로 압축합니다.

KODEX200에는 Wilder 14일 price RSI와 volume RSI도 추가합니다. 코드의 정확한 재스케일은 둘 다 $(RSI-50)/50$이며, KODEX200의 최종 기술방향은 `k_score`, `price_strength`, `volume_strength`의 단순평균입니다. 다른 세 자산은 K-score만 사용합니다. 가격은 방향을, 거래량은 그 움직임의 참여도를 확인한다는 해석입니다.

### 기술신호는 새 기대수익을 더하지 않고 거시전망의 확신을 조절한다

\[
confidence_i=clip\left[\frac12\{1+sign(\mu_i-\bar\mu)tech_i\},0,1\right]
\]

\[
\mu_{filtered,i}=\bar\mu+confidence_i(\mu_i-\bar\mu)
\]

거시가 평균보다 좋게 보는 자산의 가격추세도 좋으면 상대 기대수익을 더 신뢰합니다. 반대로 가격이 거시전망과 충돌하면 기대수익 부호를 뒤집지 않고 횡단면 평균 쪽으로 당겨 **덜 자신 있게** 만듭니다.

ATR은 방향이 아니라 위험입니다. `NATR=ATR/가격`의 causal rank가 0.9라면 variance scale은 1.9입니다. $D=diag(\sqrt{1+rank_i})$로 $D\Sigma D$를 계산하므로 해당 자산의 분산은 1.9배, 다른 자산과의 공분산은 기하평균 규모로 일관되게 조정됩니다.

In [ ]:
def numeric_ohlcv(frame: pd.DataFrame) -> pd.DataFrame:
    output = frame.copy()
    output.index = pd.to_datetime(output.index).normalize()
    columns = ["open", "high", "low", "close", "volume"]
    for column in columns:
        if column not in output:
            output[column] = np.nan
        output[column] = pd.to_numeric(output[column], errors="coerce")
    output = output[columns].sort_index()
    return output.loc[~output.index.duplicated(keep="last")]


def load_daily_asset_ohlcv() -> tuple[dict[str, pd.DataFrame], dict[str, Any]]:
    assert CACHE_DIR is not None and RAW_DIR is not None
    raw = pd.read_csv(CACHE_DIR / "regime_lightgbm_ohlcv.csv", parse_dates=["date"])
    market = {
        str(symbol): numeric_ohlcv(group.set_index("date"))
        for symbol, group in raw.groupby("symbol")
    }
    required = {"KODEX200", "GLD", "USO", "USDKRW"}
    missing = sorted(required.difference(market))
    if missing:
        raise ValueError(f"OHLCV cache is missing symbols: {missing}")
    actual = market["KODEX200"].loc[
        market["KODEX200"].index > pd.Timestamp("2009-03-31")
    ].dropna(subset=["close"])
    with sqlite3.connect(RAW_DIR / "compass.db") as connection:
        proxy = pd.read_sql(
            "select date, open, high, low, close, volume "
            "from etf_prices where symbol = ? order by date",
            connection,
            params=("1028",),
        )
    proxy["date"] = pd.to_datetime(proxy["date"])
    proxy = numeric_ohlcv(proxy.set_index("date"))
    first_actual = actual.index.min()
    nearest_position = proxy.index.get_indexer([first_actual], method="nearest")[0]
    scale = float(actual.loc[first_actual, "close"] / proxy.iloc[nearest_position]["close"])
    for column in ["open", "high", "low", "close"]:
        proxy[column] *= scale
    proxy = proxy.loc[proxy.index < first_actual].copy()
    proxy["volume_segment"] = "KOSPI200_proxy"
    actual = actual.copy()
    actual["volume_segment"] = "KODEX200_ETF"
    kodex = pd.concat([proxy, actual]).sort_index()

    bond_raw = pd.read_csv(RAW_DIR / "krx_bond_index.csv", encoding="cp949")
    bond_close = pd.to_numeric(
        bond_raw.iloc[:, 1].astype(str).str.replace(",", "", regex=False),
        errors="coerce",
    )
    bond = pd.DataFrame(
        {
            "open": bond_close.to_numpy(),
            "high": bond_close.to_numpy(),
            "low": bond_close.to_numpy(),
            "close": bond_close.to_numpy(),
            "volume": np.nan,
        },
        index=pd.to_datetime(bond_raw.iloc[:, 0]).dt.normalize(),
    ).sort_index()

    fx = market["USDKRW"]["close"].dropna().sort_index()
    foreign: dict[str, pd.DataFrame] = {}
    for asset in ["GLD", "USO"]:
        frame = market[asset].dropna(subset=["close"]).copy()
        aligned_fx = fx.reindex(frame.index, method="ffill")
        if aligned_fx.isna().any():
            raise ValueError(f"USDKRW does not cover {asset} history")
        for column in ["open", "high", "low", "close"]:
            frame[column] *= aligned_fx
        foreign[asset] = frame
    frames = {
        "KODEX200": kodex,
        "BOND": bond,
        "GLD": foreign["GLD"],
        "USO": foreign["USO"],
    }
    audit = {
        "foreign_prices_converted_to_krw": True,
        "bond_atr_uses_close_to_close_proxy": True,
        "assets": {
            asset: {
                "rows": int(len(frame)),
                "start": str(frame.index.min().date()),
                "end": str(frame.index.max().date()),
            }
            for asset, frame in frames.items()
        },
    }
    return frames, audit


def rolling_k_ratio(close: pd.Series, window: int = K_RATIO_DAYS) -> pd.Series:
    log_price = np.log(close.where(close > 0.0))
    x = np.arange(window, dtype=float)
    centered_x = x - x.mean()
    ssx = float(np.square(centered_x).sum())

    def calculate(values: np.ndarray) -> float:
        if not np.isfinite(values).all():
            return float("nan")
        centered_y = values - values.mean()
        slope = float(centered_x @ centered_y / ssx)
        residual = centered_y - slope * centered_x
        residual_variance = float(np.square(residual).sum() / (window - 2))
        slope_se = math.sqrt(max(residual_variance / ssx, 0.0))
        if slope_se <= 1e-15:
            if abs(slope) <= 1e-15:
                return 0.0
            return slope / (np.finfo(float).eps * math.sqrt(window))
        return slope / (slope_se * math.sqrt(window))

    return log_price.rolling(window, min_periods=window).apply(calculate, raw=True)


def wilder_average(values: pd.Series, period: int = WILDER_DAYS) -> pd.Series:
    values = values.astype(float)
    output = pd.Series(np.nan, index=values.index, dtype=float)
    if len(values) <= period:
        return output
    seed = values.iloc[1 : period + 1]
    if seed.notna().sum() < period:
        return output
    output.iloc[period] = float(seed.mean())
    for position in range(period + 1, len(values)):
        current = values.iloc[position]
        previous = output.iloc[position - 1]
        if np.isfinite(current) and np.isfinite(previous):
            output.iloc[position] = (previous * (period - 1) + current) / period
    return output


def average_true_range(frame: pd.DataFrame, period: int = WILDER_DAYS) -> pd.Series:
    previous_close = frame["close"].shift(1)
    true_range = pd.concat(
        [
            frame["high"] - frame["low"],
            (frame["high"] - previous_close).abs(),
            (frame["low"] - previous_close).abs(),
        ],
        axis=1,
    ).max(axis=1)
    true_range.iloc[0] = np.nan
    return wilder_average(true_range, period)


def price_rsi(close: pd.Series, period: int = WILDER_DAYS) -> pd.Series:
    change = close.diff()
    gain = wilder_average(change.clip(lower=0.0), period)
    loss = wilder_average(-change.clip(upper=0.0), period)
    return 100.0 * gain.div((gain + loss).replace(0.0, np.nan))


def volume_rsi(
    close: pd.Series, volume: pd.Series, period: int = WILDER_DAYS
) -> pd.Series:
    change = close.diff()
    up = wilder_average(volume.where(change > 0.0, 0.0), period)
    down = wilder_average(volume.where(change < 0.0, 0.0), period)
    return 100.0 * up.div((up + down).replace(0.0, np.nan))


def build_daily_technical_features(
    frames: dict[str, pd.DataFrame],
) -> dict[str, pd.DataFrame]:
    features: dict[str, pd.DataFrame] = {}
    for asset in ASSETS:
        frame = frames[asset].copy()
        output = pd.DataFrame(index=frame.index)
        output["k_ratio"] = rolling_k_ratio(frame["close"])
        output["k_score"] = output["k_ratio"] / (1.0 + output["k_ratio"].abs())
        output["atr"] = average_true_range(frame)
        output["natr"] = output["atr"] / frame["close"]
        output["atr_percentile"] = causal_expanding_midrank(output["natr"])
        if asset == "KODEX200":
            output["price_rsi"] = price_rsi(frame["close"])
            parts = [
                volume_rsi(segment["close"], segment["volume"])
                for _, segment in frame.groupby("volume_segment", sort=False)
            ]
            output["volume_rsi"] = pd.concat(parts).sort_index()
            output["price_strength"] = (output["price_rsi"] - 50.0) / 50.0
            output["volume_strength"] = (output["volume_rsi"] - 50.0) / 50.0
            output["technical_direction"] = output[
                ["k_score", "price_strength", "volume_strength"]
            ].mean(axis=1)
        else:
            output["technical_direction"] = output["k_score"]
        features[asset] = output.replace([np.inf, -np.inf], np.nan)
    return features


def build_monthly_technical_signals(
    target_months: pd.PeriodIndex,
    daily_features: dict[str, pd.DataFrame],
) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for target_month in target_months:
        signal_month = target_month - 1
        month_end = signal_month.to_timestamp("M")
        row: dict[str, Any] = {
            "target_month": target_month,
            "technical_signal_month": signal_month,
        }
        complete = True
        for asset in ASSETS:
            required = [
                "k_ratio", "k_score", "natr", "atr_percentile", "technical_direction"
            ]
            if asset == "KODEX200":
                required += ["price_rsi", "volume_rsi", "price_strength", "volume_strength"]
            known = daily_features[asset].loc[:month_end].dropna(subset=required)
            if known.empty:
                complete = False
                break
            current = known.iloc[-1]
            row[f"technical_signal_date_{asset}"] = known.index[-1]
            for column in required:
                row[f"{column}_{asset}"] = float(current[column])
        if complete:
            rows.append(row)
    monthly = pd.DataFrame(rows).set_index("target_month")
    monthly.index = pd.PeriodIndex(monthly.index, freq="M")
    return monthly


def apply_technical_inputs(
    macro_expected_return: np.ndarray,
    covariance: np.ndarray,
    signal: pd.Series,
) -> dict[str, Any]:
    macro = np.asarray(macro_expected_return, dtype=float)
    neutral = float(macro.mean())
    macro_direction = np.sign(macro - neutral)
    technical_direction = np.array(
        [float(signal[f"technical_direction_{asset}"]) for asset in ASSETS]
    )
    confidence = np.clip(
        0.5 * (1.0 + macro_direction * technical_direction), 0.0, 1.0
    )
    filtered_macro = neutral + confidence * (macro - neutral)
    atr_percentile = np.array(
        [float(signal[f"atr_percentile_{asset}"]) for asset in ASSETS]
    )
    variance_scale = 1.0 + np.clip(atr_percentile, 0.0, 1.0)
    scaling = np.diag(np.sqrt(variance_scale))
    return {
        "filtered_macro_expected_return": filtered_macro,
        "adjusted_covariance": scaling @ covariance @ scaling,
        "macro_confidence": confidence,
        "atr_variance_scale": variance_scale,
    }


## 9. EPS·밸류에이션·신용 입력

Stage35의 기준 기대수익을 재현합니다.

- KODEX200 μ: 1개월 선행 EPS revision과 `1/PER−국고채10년` 밸류에이션 갭
- 신용위험: `AA-회사채3년−국고채3년`의 20거래일 확대
- 인과 보정: 최소 60개월 과거만 이용한 비음수 단변량 기울기

신용스프레드 확대는 주식 stress μ를 확인하고 주식 공분산 축을 키웁니다. 실패한 신호를 사후에 부호반전하지 않습니다.

### EPS revision: 기업이익 전망이 좋아지는가

Forward EPS 절대수준보다 애널리스트의 1개월 revision을 사용합니다. 전망치가 100→105→112라면 이익기대가 계속 상향되는 상태입니다. 현재 revision은 과거 60개월 이상으로 만든 causal z-score로 표준화하고, 과거 월수익과의 expanding 단변량 slope로 월 기대수익 단위에 맞춥니다.

\[
\beta^*=\max(0,\hat\beta),\qquad
\Delta\mu_{EPS}=\beta^* z_{EPS}
\]

과거에 도움이 없었으면 계수를 0으로 두며, 음의 부호로 뒤집어 새 규칙을 만들지 않습니다.

### 밸류에이션: PER만 보지 않고 채권수익률과 비교한다

\[
EYG=\frac{1}{Forward\ PER}-y_{10Y}
\]

PER 10배의 earnings yield는 10%입니다. 국고채 10년이 3%면 gap은 7%지만 국채가 8%면 2%뿐입니다. 같은 PER라도 채권 대안수익률이 높으면 주식의 상대매력이 낮다는 뜻입니다. EYG는 완전히 관측된 과거 12개월 선행수익만 이용해 slope를 추정하고 월 단위로 12분의 1하여 KODEX200 `μ`에 더합니다.

### 신용스프레드: 금융시장의 혈압

\[
CreditSpread=y_{AA-,3Y}-y_{Gov,3Y}
\]

20거래일 확대분의 60개월 causal rank를 $q_C$라 하면, 실제 코드는 주식 stress 조정에 `2q_C`를 곱하고 KODEX200 공분산 축에는 $1+q_C$를 적용합니다. 즉 신용악화는 새로운 독립 매도 알파가 아니라 기존 공포신호를 확인하고 주식 위험을 높이는 역할입니다.

이 셀에는 원문에서 소스 확인이 필요하다고 남겨졌던 정확한 산술식까지 그대로 구현되어 있습니다. `winsorization=False`, `parameter_grid=False`도 함께 기록해 사후 문턱탐색을 하지 않았음을 감사할 수 있습니다.

In [ ]:
def load_fundamental_daily() -> tuple[pd.DataFrame, dict[str, Any]]:
    assert RAW_DIR is not None
    earnings = pd.read_excel(
        RAW_DIR / "260829_fwdPE.EPS.rev.xlsx",
        header=13,
        usecols=range(5),
        engine="openpyxl",
    )
    earnings.columns = [
        "date", "forward_pe_12m", "forward_eps_12m",
        "eps_revision_1w_pct", "eps_revision_1m_pct",
    ]
    earnings["date"] = pd.to_datetime(earnings["date"], errors="coerce")
    earnings = earnings.dropna(subset=["date"]).set_index("date").sort_index()
    earnings = earnings.apply(pd.to_numeric, errors="coerce")
    earnings = earnings.loc[~earnings.index.duplicated(keep="last")]
    positive_eps = earnings["forward_eps_12m"].where(earnings["forward_eps_12m"] > 0.0)
    earnings["computed_eps_revision_21d_pct"] = np.log(positive_eps).diff(21) * 100.0

    credit = pd.read_excel(
        RAW_DIR / "260829_국고채.회사채.xlsx",
        header=None,
        skiprows=14,
        usecols=range(11),
        engine="openpyxl",
    )
    credit.columns = [
        "date", "ktb_1y_pct", "ktb_2y_pct", "ktb_3y_pct", "ktb_5y_pct",
        "ktb_10y_pct", "ktb_20y_pct", "ktb_30y_pct", "ktb_50y_pct",
        "corp_aa_minus_3y_pct", "corp_bbb_minus_3y_pct",
    ]
    credit["date"] = pd.to_datetime(credit["date"], errors="coerce")
    credit = credit.dropna(subset=["date"]).set_index("date").sort_index()
    credit = credit.apply(pd.to_numeric, errors="coerce")
    credit = credit.loc[~credit.index.duplicated(keep="last")]
    credit["aa_credit_spread_pctpt"] = (
        credit["corp_aa_minus_3y_pct"] - credit["ktb_3y_pct"]
    )
    credit["aa_spread_widening_20d_pctpt"] = credit[
        "aa_credit_spread_pctpt"
    ].diff(CREDIT_CHANGE_DAYS)
    daily = earnings.join(credit, how="outer")
    daily["earnings_yield_gap"] = (
        1.0 / daily["forward_pe_12m"].where(daily["forward_pe_12m"] > 0.0)
        - daily["ktb_10y_pct"] / 100.0
    )
    audit = {
        "earnings_first_valid": str(daily["eps_revision_1m_pct"].dropna().index.min().date()),
        "credit_first_valid": str(daily["aa_spread_widening_20d_pctpt"].dropna().index.min().date()),
        "winsorization": False,
        "parameter_grid": False,
    }
    return daily.replace([np.inf, -np.inf], np.nan), audit


def build_monthly_fundamental_signals(daily: pd.DataFrame) -> pd.DataFrame:
    required = [
        "forward_pe_12m", "forward_eps_12m", "eps_revision_1m_pct",
        "computed_eps_revision_21d_pct", "ktb_3y_pct", "ktb_10y_pct",
        "corp_aa_minus_3y_pct", "aa_credit_spread_pctpt",
        "aa_spread_widening_20d_pctpt", "earnings_yield_gap",
    ]
    rows: list[dict[str, Any]] = []
    for signal_month, group in daily.groupby(daily.index.to_period("M")):
        complete = group.dropna(
            subset=["eps_revision_1m_pct", "aa_spread_widening_20d_pctpt", "earnings_yield_gap"]
        )
        if complete.empty:
            continue
        current = complete.iloc[-1]
        rows.append(
            {
                "target_month": signal_month + 1,
                "fundamental_signal_month": signal_month,
                "fundamental_signal_date": complete.index[-1],
                **{column: float(current[column]) for column in required},
            }
        )
    signals = pd.DataFrame(rows).set_index("target_month").sort_index()
    signals.index = pd.PeriodIndex(signals.index, freq="M")
    signals["eps_revision_z"] = causal_zscore(signals["eps_revision_1m_pct"])
    signals["credit_widening_z"] = causal_zscore(signals["aa_spread_widening_20d_pctpt"])
    signals["credit_easing_z"] = -signals["credit_widening_z"]
    signals["valuation_gap_z"] = causal_zscore(signals["earnings_yield_gap"])
    stress_rank = causal_expanding_midrank(signals["aa_spread_widening_20d_pctpt"])
    prior_count = (
        signals["aa_spread_widening_20d_pctpt"].notna().shift(1)
        .fillna(False).astype(int).cumsum()
    )
    signals["credit_stress_rank"] = stress_rank.where(prior_count >= MIN_CAUSAL_MONTHS)
    signals["credit_stress_multiplier"] = 2.0 * signals["credit_stress_rank"]
    return signals.replace([np.inf, -np.inf], np.nan)


def nonnegative_univariate_slope(feature: pd.Series, target: pd.Series) -> float:
    complete = pd.concat([feature, target], axis=1).dropna()
    if len(complete) < MIN_CAUSAL_MONTHS:
        return 0.0
    x = complete.iloc[:, 0].to_numpy(dtype=float)
    y = complete.iloc[:, 1].to_numpy(dtype=float)
    x_std = float(x.std(ddof=1))
    if not np.isfinite(x_std) or x_std <= 0.0:
        return 0.0
    x = (x - x.mean()) / x_std
    denominator = float(x @ x)
    if denominator <= 0.0:
        return 0.0
    raw = float(x @ (y - y.mean()) / denominator)
    return max(raw, 0.0)


def forward_compound(series: pd.Series, horizon: int) -> pd.Series:
    legs = [series.shift(-offset) for offset in range(horizon)]
    frame = pd.concat(legs, axis=1)
    valid = frame.notna().all(axis=1)
    return frame.add(1.0).prod(axis=1).sub(1.0).where(valid)


def add_causal_return_calibration(
    signals: pd.DataFrame, equity_returns: pd.Series
) -> pd.DataFrame:
    output = signals.copy()
    forward_12m_return = forward_compound(equity_returns, 12)
    rows: list[dict[str, Any]] = []
    for month in output.index:
        history = output.index[output.index < month].intersection(equity_returns.index)
        eps_feature = output.loc[history, "eps_revision_1m_pct"]
        credit_feature = -output.loc[history, "aa_spread_widening_20d_pctpt"]
        target = equity_returns.loc[history]
        eps_slope = nonnegative_univariate_slope(eps_feature, target)
        valuation_history = output.index[output.index <= month - 12].intersection(
            forward_12m_return.index
        )
        valuation_feature = output.loc[valuation_history, "earnings_yield_gap"]
        valuation_target = forward_12m_return.loc[valuation_history]
        valuation_slope = nonnegative_univariate_slope(valuation_feature, valuation_target)
        rows.append(
            {
                "target_month": month,
                "calibration_observations": int(
                    pd.concat([eps_feature, credit_feature, target], axis=1).dropna().shape[0]
                ),
                "eps_calibration_slope": eps_slope,
                "valuation_calibration_slope_12m": valuation_slope,
                "eps_mu_adjustment_KODEX200": eps_slope * float(output.loc[month, "eps_revision_z"]),
                "valuation_mu_adjustment_KODEX200": (
                    valuation_slope * float(output.loc[month, "valuation_gap_z"]) / 12.0
                ),
            }
        )
    calibrated = pd.DataFrame(rows).set_index("target_month")
    calibrated.index = pd.PeriodIndex(calibrated.index, freq="M")
    return output.join(calibrated)


## 10. Stage36의 신규 입력: GVZ→GLD, OVX→USO

FRED 형식 CSV에서 양수 지수값만 읽고 현재 이전 252개 유효관측이 쌓인 뒤 활성화합니다. 목표월 `t`에는 `t-1` 월말까지의 마지막 값만 사용합니다.

\[
m_G=1+q_{GVZ},\quad m_O=1+q_{OVX}
\]

비활성 기간의 배수는 1입니다. 출시 전 값을 실현변동성이나 다른 지수로 backfill하지 않습니다.

### ATR과 GVZ/OVX는 무엇이 다른가

- ATR: 실제 가격에서 이미 나타난 최근 변동성으로 네 자산 모두를 측정합니다.
- GVZ/OVX: 옵션시장에서 금·원유의 앞으로의 변동성에 붙은 보험가격을 측정합니다.

따라서 Stage36의 질문은 “GVZ가 높으면 금이 떨어질까?”가 아니라 “옵션시장이 금 위험을 비싸게 보고 있을 때 동일한 금 비중을 평소와 같은 위험으로 계산해도 되는가?”입니다.

GVZ rank가 0.8이면 GLD variance multiplier는 1.8, OVX rank가 0.3이면 USO multiplier는 1.3입니다. 센서가 비활성인 초기구간에는 무조건 1입니다. 순위문턱을 여러 개 시험하지 않고 전체 0~1 순위를 연속적으로 쓰므로 경계 바로 양쪽의 관측이 전혀 다른 의사결정을 만드는 것을 피합니다.

**중요한 제한:** 이 셀은 신호만 만듭니다. GLD·USO의 기대수익 조정 열은 뒤의 optimizer에서 명시적으로 0으로 저장됩니다.

In [ ]:
def load_fred_series(path: Path, value_column: str) -> pd.Series:
    raw = pd.read_csv(path)
    required = {"observation_date", value_column}
    missing = required.difference(raw.columns)
    if missing:
        raise ValueError(f"{path.name} is missing: {sorted(missing)}")
    raw["observation_date"] = pd.to_datetime(raw["observation_date"], errors="coerce")
    raw[value_column] = pd.to_numeric(raw[value_column], errors="coerce")
    series = raw.dropna(subset=["observation_date"]).set_index("observation_date")[value_column].sort_index()
    series = series.loc[~series.index.duplicated(keep="last")]
    return series.where(series > 0.0)


def load_asset_implied_volatility_daily() -> tuple[pd.DataFrame, dict[str, Any]]:
    assert RAW_DIR is not None
    gvz = load_fred_series(RAW_DIR / "GVZCLS.csv", "GVZCLS").rename("gvz")
    ovx = load_fred_series(RAW_DIR / "OVXCLS.csv", "OVXCLS").rename("ovx")
    daily = pd.concat([gvz, ovx], axis=1).sort_index()
    for sensor in ("gvz", "ovx"):
        rank, count = rank_after_prior_history(daily[sensor])
        daily[f"{sensor}_causal_rank"] = rank
        daily[f"{sensor}_prior_valid_observations"] = count
    audit = {
        "gvz_first_valid": str(gvz.dropna().index.min().date()),
        "gvz_last_valid": str(gvz.dropna().index.max().date()),
        "gvz_valid_observations": int(gvz.notna().sum()),
        "ovx_first_valid": str(ovx.dropna().index.min().date()),
        "ovx_last_valid": str(ovx.dropna().index.max().date()),
        "ovx_valid_observations": int(ovx.notna().sum()),
        "minimum_prior_observations": MIN_SENSOR_HISTORY,
        "directional_mu_effect": False,
    }
    return daily.replace([np.inf, -np.inf], np.nan), audit


def build_monthly_asset_volatility_signals(
    daily: pd.DataFrame, target_months: pd.PeriodIndex
) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for target_month in target_months:
        signal_month = target_month - 1
        month_end = signal_month.to_timestamp("M")
        row: dict[str, Any] = {
            "target_month": target_month,
            "asset_vol_signal_month": signal_month,
        }
        for sensor, asset in (("gvz", "GLD"), ("ovx", "USO")):
            known = daily.loc[
                :month_end,
                [sensor, f"{sensor}_causal_rank", f"{sensor}_prior_valid_observations"],
            ].dropna(subset=[sensor])
            if known.empty:
                value, rank, count, signal_date = np.nan, np.nan, 0, pd.NaT
            else:
                current = known.iloc[-1]
                value = float(current[sensor])
                rank = float(current[f"{sensor}_causal_rank"])
                count = int(current[f"{sensor}_prior_valid_observations"])
                signal_date = known.index[-1]
            active = bool(np.isfinite(rank) and count >= MIN_SENSOR_HISTORY)
            row.update(
                {
                    f"{sensor}_signal_date": signal_date,
                    f"{sensor}_level": value,
                    f"{sensor}_causal_rank": rank,
                    f"{sensor}_prior_valid_observations": count,
                    f"{sensor}_active": active,
                    f"{sensor}_{asset.lower()}_variance_multiplier": 1.0 + rank if active else 1.0,
                }
            )
        rows.append(row)
    signals = pd.DataFrame(rows).set_index("target_month")
    signals.index = pd.PeriodIndex(signals.index, freq="M")
    return signals.replace([np.inf, -np.inf], np.nan)


## 11. DΣD 오버레이와 SLSQP

Stage36은 기대수익을 바꾸지 않고 공분산만 바꿉니다.

\[
D=diag(1,1,\sqrt{m_G},\sqrt{m_O}),\qquad
\Sigma_{36}=D\Sigma_{35}D,\qquad \mu_{36}=\mu_{35}
\]

최적화 목적은 다음 월별 효용 최대화입니다.

\[
w'\mu-\tfrac12w'\Sigma w-
E[\min(R_{hist}w,0)^2]-C(w,w^{pre})
\]

제약은 비중합 1, 각 비중 0~1, 연변동성≤13%, 역사적 CDaR(90%)≥-16%입니다. SLSQP가 실패할 때만 같은 제약의 최소분산 fallback을 사용합니다.

### `μ35`가 실제 코드에서 조립되는 정확한 순서

1. `estimate_conditional_moments`에서 `macro_expected_return`과 `stress_return_adjustment`를 분리해 받습니다.
2. 거시 기대수익만 K-ratio·RSI confidence filter에 통과시킵니다.
3. KODEX200의 filtered macro에 EPS·valuation 조정을 더합니다.
4. 신용 stress multiplier를 KODEX200의 stress adjustment에만 곱합니다.
5. 마지막으로 `expected_return = filtered_macro + stress_adjustment`를 계산합니다.

개념식은 다음과 같습니다.

\[
\mu_{35}=Filter_{technical}(\mu_{macro})
+\Delta\mu_{EPS}+\Delta\mu_{valuation}
+Confirm_{credit}(\Delta\mu_{stress/recovery})
\]

### `Σ35`와 `Σ36`의 정확한 순서

\[
\Sigma_{macro/stress}
\rightarrow D_{ATR}\Sigma D_{ATR}
\rightarrow D_{credit}\Sigma D_{credit}
\rightarrow D_{GVZ,OVX}\Sigma D_{GVZ,OVX}
\]

Stage36은 마지막 화살표 하나만 추가합니다. `asset_scaling` 전후에 `expected_return`을 변경하는 코드가 없고 결과에도 `gvz_mu_adjustment_GLD=0`, `ovx_mu_adjustment_USO=0`을 기록합니다.

### SLSQP가 최대화하는 것

\[
U(w)=w'\mu-\frac12w'\Sigma w
-E[\min(R_{hist}w,0)^2]-C(w,w^{pre})
\]

- $w'\mu$: 기대수익 보상
- $\frac12w'\Sigma w$: 현재 추정 변동성 벌점
- 하방반분산: 과거 손실구간만 제곱해 부과하는 추가 벌점
- 거래비용: 현재 사전비중에서 새 비중으로 이동하는 마찰

제약은 $\sum w_i=1$, $0\le w_i\le1$, 연변동성 13% 이하, 과거 90% CDaR가 -16%보다 나쁘지 않을 것입니다. “단일자산 과반금지”는 없지만 비중합 1과 개별 0~1 때문에 레버리지와 공매도는 불가능합니다.

SLSQP는 목적함수의 기울기와 제약면을 함께 따라가는 비선형 최적화법입니다. 실패했을 때만 동일 제약의 최소분산 문제로 fallback합니다. `maxiter`와 `ftol`은 경제가설이 아니라 해를 얼마나 오래·정밀하게 찾을지 정한 수치설정입니다.

In [ ]:
def project_to_long_only_simplex(weights: np.ndarray) -> np.ndarray:
    values = np.asarray(weights, dtype=float)
    if not np.isfinite(values).all():
        raise ValueError("Weights must be finite")
    ordered = np.sort(values)[::-1]
    cumulative = np.cumsum(ordered) - 1.0
    candidates = ordered - cumulative / np.arange(1, len(values) + 1) > 0
    rho = int(np.flatnonzero(candidates)[-1])
    threshold = cumulative[rho] / float(rho + 1)
    projected = np.maximum(values - threshold, 0.0)
    return projected / projected.sum()


def cdar(returns: np.ndarray, alpha: float = CDAR_CONFIDENCE) -> float:
    wealth = np.cumprod(1.0 + returns)
    drawdown = wealth / np.maximum.accumulate(np.r_[1.0, wealth])[-len(wealth):] - 1.0
    count = max(1, int(math.ceil((1.0 - alpha) * len(drawdown))))
    return float(np.mean(np.sort(drawdown)[:count]))


def expected_transaction_cost(weights: np.ndarray, pretrade: np.ndarray) -> float:
    change = weights - pretrade
    smooth_absolute_change = np.sqrt(change**2 + NUMERICAL_EPSILON)
    trade_cost = float(smooth_absolute_change.sum()) * DOMESTIC_TRADE_COST
    foreign_indices = [GOLD_INDEX, OIL_INDEX]
    foreign_change = float(change[foreign_indices].sum())
    fx_cost = math.sqrt(foreign_change**2 + NUMERICAL_EPSILON) * FOREIGN_WEIGHT_CHANGE_COST
    return trade_cost + fx_cost


def mode_uses_gvz(mode: str) -> bool:
    return mode in {"gvz_gold_risk", "gvz_ovx_asset_risk"}


def mode_uses_ovx(mode: str) -> bool:
    return mode in {"ovx_oil_risk", "gvz_ovx_asset_risk"}


def solve_weights(
    history: pd.DataFrame,
    historical_probabilities: pd.DataFrame,
    current_probabilities: pd.Series,
    historical_stress: pd.Series,
    current_stress: float,
    historical_recovery: pd.Series,
    current_recovery: float,
    technical_signal: pd.Series,
    fundamental_signal: pd.Series,
    asset_vol_signal: pd.Series,
    pretrade: np.ndarray,
    mode: str,
) -> tuple[np.ndarray, dict[str, Any]]:
    _, base_covariance, moment_detail = estimate_conditional_moments(
        history,
        historical_probabilities,
        current_probabilities,
        historical_stress,
        current_stress,
        historical_recovery,
        current_recovery,
    )
    macro_expected_return = np.asarray(
        moment_detail["macro_expected_monthly_return"], dtype=float
    )
    stress_adjustment = np.asarray(
        moment_detail["stress_return_adjustment"], dtype=float
    ).copy()
    technical = apply_technical_inputs(
        macro_expected_return, base_covariance, technical_signal
    )
    filtered_macro = np.asarray(
        technical["filtered_macro_expected_return"], dtype=float
    ).copy()

    eps_mu = float(fundamental_signal["eps_mu_adjustment_KODEX200"])
    valuation_mu = float(fundamental_signal["valuation_mu_adjustment_KODEX200"])
    credit_stress_multiplier = float(fundamental_signal["credit_stress_multiplier"])
    stress_adjustment[EQUITY_INDEX] *= credit_stress_multiplier
    filtered_macro[EQUITY_INDEX] += eps_mu + valuation_mu
    expected_return = filtered_macro + stress_adjustment

    covariance = np.asarray(technical["adjusted_covariance"], dtype=float)
    credit_variance_multiplier = 1.0 + float(fundamental_signal["credit_stress_rank"])
    credit_scaling = np.eye(len(ASSETS), dtype=float)
    credit_scaling[EQUITY_INDEX, EQUITY_INDEX] = math.sqrt(credit_variance_multiplier)
    covariance = credit_scaling @ covariance @ credit_scaling

    gvz_multiplier = (
        float(asset_vol_signal["gvz_gld_variance_multiplier"])
        if mode_uses_gvz(mode)
        else 1.0
    )
    ovx_multiplier = (
        float(asset_vol_signal["ovx_uso_variance_multiplier"])
        if mode_uses_ovx(mode)
        else 1.0
    )
    asset_scaling = np.eye(len(ASSETS), dtype=float)
    asset_scaling[GOLD_INDEX, GOLD_INDEX] = math.sqrt(gvz_multiplier)
    asset_scaling[OIL_INDEX, OIL_INDEX] = math.sqrt(ovx_multiplier)
    covariance = asset_scaling @ covariance @ asset_scaling

    common = history.index.intersection(historical_stress.dropna().index)
    historical_returns = history.loc[common, ASSETS].to_numpy(dtype=float)
    initial = (
        project_to_long_only_simplex(pretrade)
        if np.isfinite(pretrade).all() and pretrade.sum() > 0.99
        else np.repeat(1.0 / len(ASSETS), len(ASSETS))
    )

    def portfolio_values(weights: np.ndarray) -> dict[str, float]:
        monthly_return = float(weights @ expected_return)
        monthly_variance = max(float(weights @ covariance @ weights), 0.0)
        realized_history = historical_returns @ weights
        downside_semivariance = float(np.mean(np.minimum(realized_history, 0.0) ** 2))
        transaction_cost = expected_transaction_cost(weights, pretrade)
        utility = (
            monthly_return
            - 0.5 * monthly_variance
            - downside_semivariance
            - transaction_cost
        )
        return {
            "expected_monthly_return": monthly_return,
            "expected_monthly_variance": monthly_variance,
            "estimated_transaction_cost": transaction_cost,
            "monthly_utility": utility,
        }

    def annual_volatility(weights: np.ndarray) -> float:
        return math.sqrt(max(float(weights @ covariance @ weights), 0.0) * 12.0)

    constraints = [
        {"type": "eq", "fun": lambda weights: float(weights.sum() - 1.0)},
        {
            "type": "ineq",
            "fun": lambda weights: CATASTROPHE_ANNUAL_VOLATILITY - annual_volatility(weights),
        },
        {
            "type": "ineq",
            "fun": lambda weights: CATASTROPHE_CDAR
            + cdar(historical_returns @ weights, CDAR_CONFIDENCE),
        },
    ]
    result = minimize(
        lambda weights: -portfolio_values(weights)["monthly_utility"],
        initial,
        method="SLSQP",
        bounds=UNCONSTRAINED_LONG_ONLY_BOUNDS,
        constraints=constraints,
        options={"maxiter": SLSQP_MAX_ITERATIONS, "ftol": SLSQP_TOLERANCE},
    )
    used_fallback = False
    if result.success and np.isfinite(result.x).all():
        weights = project_to_long_only_simplex(result.x)
    else:
        fallback = minimize(
            lambda weights: float(weights @ covariance @ weights),
            initial,
            method="SLSQP",
            bounds=UNCONSTRAINED_LONG_ONLY_BOUNDS,
            constraints=constraints,
            options={"maxiter": SLSQP_MAX_ITERATIONS, "ftol": SLSQP_TOLERANCE},
        )
        if not fallback.success or not np.isfinite(fallback.x).all():
            raise RuntimeError(f"Both SLSQP solves failed: {result.message}; {fallback.message}")
        result = fallback
        weights = project_to_long_only_simplex(fallback.x)
        used_fallback = True

    values = portfolio_values(weights)
    annual_vol = annual_volatility(weights)
    historical_cdar = cdar(historical_returns @ weights, CDAR_CONFIDENCE)
    return weights, {
        **values,
        "policy": f"Stage36_{mode}",
        "solver_success": bool(result.success),
        "used_fallback": used_fallback,
        "solver_status": int(result.status),
        "solver_iterations": int(result.nit),
        "expected_annual_volatility": annual_vol,
        "historical_cdar": historical_cdar,
        "sum_error": abs(float(weights.sum()) - 1.0),
        "volatility_slack": CATASTROPHE_ANNUAL_VOLATILITY - annual_vol,
        "cdar_slack": CATASTROPHE_CDAR + historical_cdar,
        "eps_mu_adjustment_KODEX200": eps_mu,
        "valuation_mu_adjustment_KODEX200": valuation_mu,
        "credit_stress_confirmation_multiplier": credit_stress_multiplier,
        "credit_equity_variance_multiplier": credit_variance_multiplier,
        "gvz_gold_variance_multiplier": gvz_multiplier,
        "ovx_oil_variance_multiplier": ovx_multiplier,
        "gvz_mu_adjustment_GLD": 0.0,
        "ovx_mu_adjustment_USO": 0.0,
        "expected_mu_GLD": float(expected_return[GOLD_INDEX]),
        "expected_mu_USO": float(expected_return[OIL_INDEX]),
    }


## 12. 월별 비용 차감 백테스트

월 `t`보다 앞선 수익만 추정에 넣고, SLSQP 비중에 당월 실현수익을 적용합니다. 전체 비중변화의 15bp와 GLD·USO 순비중 변화의 추가 5bp를 차감합니다.

월말의 다음 사전비중은

\[
w^{pre}_{i,t+1}=\frac{w_{i,t}(1+r_{i,t})}{1+w_t'r_t}
\]

로 계산하므로 가격변동으로 자연스럽게 떠밀린 비중과 새 목표비중 사이의 실제 매매량을 비용에 반영합니다.

### 직전 비중도 중요한 입력변수다

예측변수만으로 최적화하면 매달 100% 갈아타는 해가 좋아 보일 수 있습니다. 실제로는 지난달 목표비중이 자산수익률 때문에 자연스럽게 변한 `pretrade` 비중에서 출발해야 합니다.

예를 들어 지난달 GLD 목표가 30%였더라도 GLD만 크게 오르면 이번 달 리밸런싱 직전 비중은 30%보다 커집니다. 코드는 이 drift를 계산한 뒤 새 목표와의 차이에 비용을 매깁니다. 첫 진입에는 전체 절대변화량, 이후 보고용 turnover에는 양방향 매매의 중복계산을 피하려고 절반을 사용합니다. 실제 비용은 국내 15bp와 해외 순비중 변화 5bp를 모두 차감합니다.

각 반복에서 `history = returns[index < month]`로 잘라 현재월 실현수익이 추정과 최적화에 들어가지 않도록 합니다. 비중을 먼저 결정한 뒤에만 `returns.loc[month]`를 적용하는 순서를 코드에서 확인할 수 있습니다.

In [ ]:
def run_backtest(
    returns: pd.DataFrame,
    probabilities: pd.DataFrame,
    stress_signals: pd.DataFrame,
    technical_signals: pd.DataFrame,
    fundamental_signals: pd.DataFrame,
    asset_vol_signals: pd.DataFrame,
    mode: str,
) -> pd.DataFrame:
    required_fundamental = [
        "eps_mu_adjustment_KODEX200",
        "valuation_mu_adjustment_KODEX200",
        "credit_stress_multiplier",
        "credit_stress_rank",
    ]
    months = returns.index.intersection(probabilities.index)
    months = months.intersection(stress_signals.index)
    months = months.intersection(technical_signals.index)
    months = months.intersection(asset_vol_signals.index)
    months = months.intersection(
        fundamental_signals.dropna(subset=required_fundamental).index
    )
    months = months[(months >= FULL_START) & (months <= RESEARCH_END)]
    rows: list[dict[str, Any]] = []
    pretrade = np.zeros(len(ASSETS), dtype=float)
    first_trade = True
    nav, peak = 1.0, 1.0
    for month in months:
        history = returns.loc[returns.index < month, ASSETS]
        if len(history) < ONE_CALENDAR_YEAR:
            continue
        probability = probabilities.loc[month]
        asset_vol_signal = asset_vol_signals.loc[month]
        weights, detail = solve_weights(
            history,
            probabilities.loc[probabilities.index < month],
            probability,
            stress_signals.loc[stress_signals.index < month, "stress_score"],
            float(stress_signals.loc[month, "stress_score"]),
            stress_signals.loc[stress_signals.index < month, "recovery_score"],
            float(stress_signals.loc[month, "recovery_score"]),
            technical_signals.loc[month],
            fundamental_signals.loc[month],
            asset_vol_signal,
            pretrade,
            mode,
        )
        change = weights - pretrade
        turnover = float(np.abs(change).sum()) if first_trade else 0.5 * float(np.abs(change).sum())
        trade_cost = float(np.abs(change).sum()) * DOMESTIC_TRADE_COST
        fx_cost = abs(float(change[[GOLD_INDEX, OIL_INDEX]].sum())) * FOREIGN_WEIGHT_CHANGE_COST
        asset_return = returns.loc[month, ASSETS].to_numpy(dtype=float)
        gross_return = float(weights @ asset_return)
        net_return = gross_return - trade_cost - fx_cost
        nav *= 1.0 + net_return
        peak = max(peak, nav)
        pretrade = weights * (1.0 + asset_return) / (1.0 + gross_return)
        first_trade = False
        rows.append(
            {
                "month": month,
                "asset_vol_signal_month": asset_vol_signal["asset_vol_signal_month"],
                "gvz_signal_date": asset_vol_signal["gvz_signal_date"],
                "ovx_signal_date": asset_vol_signal["ovx_signal_date"],
                "gvz_level": float(asset_vol_signal["gvz_level"]),
                "ovx_level": float(asset_vol_signal["ovx_level"]),
                "gvz_causal_rank": float(asset_vol_signal["gvz_causal_rank"]),
                "ovx_causal_rank": float(asset_vol_signal["ovx_causal_rank"]),
                "gvz_active": bool(asset_vol_signal["gvz_active"]),
                "ovx_active": bool(asset_vol_signal["ovx_active"]),
                "return": net_return,
                "gross_return": gross_return,
                "nav": nav,
                "drawdown": nav / peak - 1.0,
                "turnover": turnover,
                "trade_cost": trade_cost,
                "fx_cost": fx_cost,
                **{f"w_{asset}": float(weights[i]) for i, asset in enumerate(ASSETS)},
                **detail,
            }
        )
    output = pd.DataFrame(rows).set_index("month")
    output.index = pd.PeriodIndex(output.index, freq="M")
    return output


## 13. 성과지표와 paired circular block bootstrap

CAGR, 월수익 표준편차×√12, 무위험수익률 0 기준 Sharpe, 월말 NAV MDD를 계산합니다. 시계열 군집을 보존하기 위해 Stage35와 Stage36의 동일 월을 12개월 원형 블록으로 묶어 2,000회 재표집합니다. 이는 위험효율 개선의 불확실성을 보여주는 진단이지 새로운 최적화 파라미터가 아닙니다.

In [ ]:
def performance_summary(returns: pd.Series) -> pd.Series:
    values = pd.Series(returns).dropna()
    wealth = (1.0 + values).cumprod()
    years = len(values) / 12.0
    cagr = wealth.iloc[-1] ** (1.0 / years) - 1.0 if years > 0 else np.nan
    volatility = values.std(ddof=1) * math.sqrt(12.0)
    sharpe = (
        values.mean() / values.std(ddof=1) * math.sqrt(12.0)
        if values.std(ddof=1) > 0
        else np.nan
    )
    drawdown = wealth / wealth.cummax() - 1.0
    mdd = float(drawdown.min())
    downside = np.sqrt(np.mean(np.minimum(values, 0.0) ** 2)) * math.sqrt(12.0)
    return pd.Series(
        {
            "Months": len(values),
            "CAGR": cagr,
            "Volatility": volatility,
            "Sharpe": sharpe,
            "Sortino": values.mean() * 12.0 / downside if downside > 0 else np.nan,
            "MDD": mdd,
            "Calmar": cagr / abs(mdd) if mdd < 0 else np.nan,
            "FinalMultiple": wealth.iloc[-1],
            "PositiveMonths": (values > 0.0).mean(),
        }
    )


def metric_row(
    name: str, path: pd.DataFrame, period: str, start: pd.Period, end: pd.Period
) -> dict[str, Any]:
    view = path.loc[start:end]
    metrics = performance_summary(view["return"])
    return {
        "Strategy": name,
        "Period": period,
        "Start": str(view.index.min()),
        "End": str(view.index.max()),
        **{key: float(value) for key, value in metrics.items()},
        "AvgTurnover": float(view["turnover"].mean()),
        "TotalCost": float(view[["trade_cost", "fx_cost"]].sum().sum()),
    }


def performance_table(paths: dict[str, pd.DataFrame]) -> pd.DataFrame:
    common_end = min(path.index.max() for path in paths.values())
    periods = {
        "full_2007_2026": (FULL_START, common_end),
        "common_2010_2026": (COMMON_START, common_end),
        "locked_2018_2026": (LOCKED_START, common_end),
    }
    return pd.DataFrame(
        [
            metric_row(name, path, period, start, end)
            for name, path in paths.items()
            for period, (start, end) in periods.items()
        ]
    )


def solver_summary(path: pd.DataFrame) -> dict[str, Any]:
    return {
        "months": int(len(path)),
        "successes": int(path["solver_success"].sum()),
        "fallbacks": int(path["used_fallback"].sum()),
        "maximum_weight_sum_error": float(path["sum_error"].max()),
        "minimum_volatility_slack": float(path["volatility_slack"].min()),
        "minimum_cdar_slack": float(path["cdar_slack"].min()),
    }


def return_metrics(values: np.ndarray) -> dict[str, float]:
    years = len(values) / 12.0
    nav = np.cumprod(1.0 + values)
    return {
        "CAGR": float(nav[-1] ** (1.0 / years) - 1.0),
        "Volatility": float(values.std(ddof=1) * math.sqrt(12.0)),
        "Sharpe": float(values.mean() / values.std(ddof=1) * math.sqrt(12.0)),
        "MDD": float(np.min(nav / np.maximum.accumulate(nav) - 1.0)),
    }


def paired_block_bootstrap(
    baseline: pd.Series,
    candidate: pd.Series,
    replications: int = 2000,
    block_months: int = 12,
) -> pd.DataFrame:
    common = baseline.index.intersection(candidate.index)
    base = baseline.loc[common].to_numpy(dtype=float)
    test = candidate.loc[common].to_numpy(dtype=float)
    rng = np.random.default_rng(20260829)
    rows: list[dict[str, float]] = []
    blocks_needed = math.ceil(len(common) / block_months)
    for _ in range(replications):
        starts = rng.integers(0, len(common), size=blocks_needed)
        indices = np.concatenate(
            [(start + np.arange(block_months)) % len(common) for start in starts]
        )[: len(common)]
        base_metrics = return_metrics(base[indices])
        test_metrics = return_metrics(test[indices])
        rows.append(
            {
                "delta_CAGR": test_metrics["CAGR"] - base_metrics["CAGR"],
                "delta_Sharpe": test_metrics["Sharpe"] - base_metrics["Sharpe"],
                "delta_MDD": test_metrics["MDD"] - base_metrics["MDD"],
            }
        )
    draws = pd.DataFrame(rows)
    return pd.DataFrame(
        [
            {
                "Metric": column,
                "Mean": float(draws[column].mean()),
                "P05": float(draws[column].quantile(0.05)),
                "P50": float(draws[column].quantile(0.50)),
                "P95": float(draws[column].quantile(0.95)),
                "ProbabilityPositive": float((draws[column] > 0.0).mean()),
                "Replications": replications,
                "BlockMonths": block_months,
            }
            for column in draws.columns
        ]
    )


## 14. GVZ/OVX의 미래위험 설명력

GVZ와 OVX가 자기자산의 향후 1개월 실현변동성, 1·3개월 최대낙폭 크기, 왼쪽꼬리를 설명하는지 HAC 회귀로 검사합니다. 통제변수는 자기자산 최근 1개월 수익, 21일 실현변동성, VIX6 stress, 거시 취약도입니다.

이 미래 목적변수는 검증에만 쓰이며 월별 비중 산정에는 들어가지 않습니다.

In [ ]:
def realized_volatility_signal(close: pd.Series) -> pd.Series:
    daily = np.log(close.where(close > 0.0)).diff()
    rolling = daily.rolling(21, min_periods=15).std(ddof=1) * math.sqrt(252.0)
    monthly = rolling.groupby(rolling.index.to_period("M")).last()
    monthly.index = pd.PeriodIndex(monthly.index + 1, freq="M")
    return monthly.rename("realized_vol_21d")


def forward_risk_targets(close: pd.Series) -> pd.DataFrame:
    close = close.dropna().sort_index()
    log_returns = np.log(close).diff()
    periods = pd.period_range(
        close.index.min().to_period("M"), RESEARCH_END, freq="M"
    )
    rows: list[dict[str, Any]] = []
    for period in periods:
        month_returns = log_returns.loc[
            log_returns.index.to_period("M") == period
        ].dropna()
        row: dict[str, Any] = {
            "target_month": period,
            "future_realized_vol_1m": (
                float(month_returns.std(ddof=1) * math.sqrt(252.0))
                if len(month_returns) >= 15
                else np.nan
            ),
        }
        for horizon, minimum_prices in ((1, 16), (3, 46)):
            end_period = period + horizon - 1
            before = close.loc[close.index < period.start_time]
            within = close.loc[
                (close.index >= period.start_time)
                & (close.index <= end_period.end_time)
            ]
            if before.empty or len(within) < minimum_prices - 1:
                value = np.nan
            else:
                path = pd.concat([before.iloc[[-1]], within])
                wealth = path / float(path.iloc[0])
                value = float(-(wealth / wealth.cummax() - 1.0).min())
            row[f"future_max_drawdown_{horizon}m"] = value
        rows.append(row)
    output = pd.DataFrame(rows).set_index("target_month")
    output.index = pd.PeriodIndex(output.index, freq="M")
    return output


def causal_tail_event(monthly_return: pd.Series) -> pd.Series:
    threshold = (
        monthly_return.shift(1)
        .expanding(min_periods=MIN_CAUSAL_MONTHS)
        .quantile(0.05)
    )
    return pd.Series(
        np.where(
            monthly_return.notna() & threshold.notna(),
            (monthly_return <= threshold).astype(float),
            np.nan,
        ),
        index=monthly_return.index,
        name="future_left_tail_1m",
    )


def build_asset_risk_research_frame(
    signals: pd.DataFrame,
    returns: pd.DataFrame,
    probabilities: pd.DataFrame,
    stress: pd.DataFrame,
    market: dict[str, pd.DataFrame],
) -> pd.DataFrame:
    frame = signals.copy()
    frame["vix6_stress_score"] = stress["stress_score"]
    frame["macro_fragility"] = (
        probabilities["p_Slowdown"] + probabilities["p_Stagflation"]
    )
    for sensor, asset in (("gvz", "GLD"), ("ovx", "USO")):
        prefix = asset.lower()
        close = market[asset]["close"].dropna()
        frame[f"{prefix}_recent_1m_return"] = returns[asset].shift(1)
        frame[f"{prefix}_realized_vol_21d"] = realized_volatility_signal(close)
        risk_targets = forward_risk_targets(close).rename(
            columns={
                "future_realized_vol_1m": f"{prefix}_future_realized_vol_1m",
                "future_max_drawdown_1m": f"{prefix}_future_max_drawdown_1m",
                "future_max_drawdown_3m": f"{prefix}_future_max_drawdown_3m",
            }
        )
        frame = frame.join(risk_targets, how="left")
        monthly_close = close.groupby(close.index.to_period("M")).last()
        monthly_return = monthly_close.pct_change().loc[:RESEARCH_END]
        frame[f"{prefix}_future_left_tail_1m"] = causal_tail_event(
            monthly_return
        )
    return frame.loc[FULL_START:RESEARCH_END].replace(
        [np.inf, -np.inf], np.nan
    )


def standardize(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    output = pd.DataFrame(index=frame.index)
    for column in columns:
        std = float(frame[column].std(ddof=0))
        if not np.isfinite(std) or std <= 0.0:
            raise ValueError(f"No usable variation in {column}")
        output[column] = (frame[column] - frame[column].mean()) / std
    return output


def asset_risk_predictive_regressions(frame: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    periods = {
        "available_full": (FULL_START, RESEARCH_END),
        "common_2010_2026": (COMMON_START, RESEARCH_END),
        "locked_2018_2026": (LOCKED_START, RESEARCH_END),
    }
    targets = (
        ("future_realized_vol_1m", 1),
        ("future_max_drawdown_1m", 1),
        ("future_max_drawdown_3m", 3),
        ("future_left_tail_1m", 1),
    )
    for sensor, asset in (("gvz", "GLD"), ("ovx", "USO")):
        prefix = asset.lower()
        feature = f"{sensor}_causal_rank"
        controls = [
            f"{prefix}_recent_1m_return",
            f"{prefix}_realized_vol_21d",
            "vix6_stress_score",
            "macro_fragility",
        ]
        for period_name, (start, end) in periods.items():
            view = frame.loc[start:end]
            for target_suffix, lags in targets:
                target = f"{prefix}_{target_suffix}"
                for model, predictors in (
                    ("SensorOnly", [feature]),
                    ("FullControls", [feature, *controls]),
                ):
                    complete = view[[target, *predictors]].dropna()
                    if len(complete) < 36:
                        continue
                    standardized = standardize(complete, predictors)
                    fit = sm.OLS(
                        complete[target], sm.add_constant(standardized)
                    ).fit(cov_type="HAC", cov_kwds={"maxlags": lags})
                    ic, ic_p = spearmanr(
                        complete[feature], complete[target], nan_policy="omit"
                    )
                    rows.append(
                        {
                            "Sensor": sensor.upper(),
                            "Asset": asset,
                            "Period": period_name,
                            "Target": target_suffix,
                            "Model": model,
                            "Observations": int(len(complete)),
                            "SensorStandardizedBeta": float(
                                fit.params[feature]
                            ),
                            "SensorHACPValue": float(fit.pvalues[feature]),
                            "SensorSpearmanIC": float(ic),
                            "SensorICPValue": float(ic_p),
                            "AdjustedR2": float(fit.rsquared_adj),
                            "HACLags": lags,
                        }
                    )
    return pd.DataFrame(rows)


## 15. 전체 조립과 네 경로 비교

지금까지 정의한 함수를 다음 순서로 연결합니다.

`월수익 → 거시확률 → VKOSPI/VIX6 → 기술신호 → EPS·신용 → GVZ/OVX → SLSQP`

Stage35, GVZ-only, OVX-only, GVZ+OVX를 동일 비용·동일 제약으로 실행합니다. 결과 CSV, 월별 비중, 위험회귀, bootstrap, solver audit를 `colab_outputs`에 저장합니다. 코드에 포함된 인과성·무레버리지·μ 불변 검사가 하나라도 실패하면 최종 감사 셀에서 오류가 납니다.

### 전체 알고리즘을 한 줄씩 추적하기

```text
GDP·수출·BSI → g
CPI·PPI·수입물가 → π
g, π → 네 soft-regime 확률
과거 확률가중 월수익 → μ_macro, Σ_macro
VKOSPI·VIX6 → stress/recovery adjustment
K-ratio·RSI → μ_macro confidence filter
ATR → Σ의 네 자산 위험축
EPS revision·EYG → KODEX200 μ
AA- spread → 주식 stress·Σ
= Stage35의 μ35, Σ35
GVZ→GLD, OVX→USO → Σ36만 추가 조정
SLSQP → 다음 달 long-only 무레버리지 비중
```

네 경로를 함께 계산하는 이유는 Stage36 결합결과만 보면 어느 센서가 영향을 냈는지 알기 어렵기 때문입니다. `Stage35_Frozen`은 기준, `GVZ-only`와 `OVX-only`는 개별 기여도, `GVZ+OVX`는 최종 후보입니다. 미래위험 회귀는 센서의 경제적 역할을 검사할 뿐 비중결정에는 사용되지 않습니다.

In [ ]:
def run_stage36_research(
    save: bool = True, bootstrap_replications: int = 2000
) -> dict[str, Any]:
    assert OUTPUT_DIR is not None
    returns, _ = load_monthly_asset_returns()
    probabilities, _ = build_macro_probabilities(returns)
    daily_stress = build_daily_stress_features()
    stress = build_monthly_stress_signals(returns.index, daily_stress)
    market, market_audit = load_daily_asset_ohlcv()
    technical_features = build_daily_technical_features(market)
    technical = build_monthly_technical_signals(
        returns.index, technical_features
    )
    raw_fundamental, fundamental_audit = load_fundamental_daily()
    fundamental = build_monthly_fundamental_signals(raw_fundamental)
    equity_close = market["KODEX200"]["close"].dropna()
    equity_monthly_close = equity_close.groupby(
        equity_close.index.to_period("M")
    ).last()
    calibrated = add_causal_return_calibration(
        fundamental, equity_monthly_close.pct_change()
    )
    daily_asset_vol, asset_vol_audit = load_asset_implied_volatility_daily()
    asset_vol = build_monthly_asset_volatility_signals(
        daily_asset_vol, returns.index
    )

    modes = {
        "Stage35_Frozen": "baseline_reproduction",
        "Stage36_GVZGoldRisk": "gvz_gold_risk",
        "Stage36_OVXOilRisk": "ovx_oil_risk",
        "Stage36_GVZ_OVXAssetRisk": "gvz_ovx_asset_risk",
    }
    paths = {
        name: run_backtest(
            returns,
            probabilities,
            stress,
            technical,
            calibrated,
            asset_vol,
            mode,
        )
        for name, mode in modes.items()
    }
    performance = performance_table(paths)
    boot_rows: list[pd.DataFrame] = []
    baseline = paths["Stage35_Frozen"]
    candidate = paths["Stage36_GVZ_OVXAssetRisk"]
    for period_name, start in (
        ("full_2007_2026", FULL_START),
        ("common_2010_2026", COMMON_START),
    ):
        summary = paired_block_bootstrap(
            baseline.loc[start:RESEARCH_END, "return"],
            candidate.loc[start:RESEARCH_END, "return"],
            replications=bootstrap_replications,
        )
        summary.insert(0, "Period", period_name)
        summary.insert(0, "Candidate", "Stage36_GVZ_OVXAssetRisk")
        boot_rows.append(summary)
    bootstrap = pd.concat(boot_rows, ignore_index=True)

    risk_frame = build_asset_risk_research_frame(
        asset_vol, returns, probabilities, stress, market
    )
    risk_tests = asset_risk_predictive_regressions(risk_frame)
    source_checks = {
        "signal_month_precedes_target": bool(
            (asset_vol["asset_vol_signal_month"] < asset_vol.index).all()
        ),
        "no_backfill_before_252": bool(
            asset_vol.loc[
                ~asset_vol["gvz_active"],
                "gvz_gld_variance_multiplier",
            ].eq(1.0).all()
            and asset_vol.loc[
                ~asset_vol["ovx_active"],
                "ovx_uso_variance_multiplier",
            ].eq(1.0).all()
        ),
        "variance_multipliers_between_one_and_two": bool(
            asset_vol[
                [
                    "gvz_gld_variance_multiplier",
                    "ovx_uso_variance_multiplier",
                ]
            ].ge(1.0).all().all()
            and asset_vol[
                [
                    "gvz_gld_variance_multiplier",
                    "ovx_uso_variance_multiplier",
                ]
            ].le(2.0).all().all()
        ),
        "no_leverage_long_only_sum_to_one": bool(
            all(
                np.allclose(path[WEIGHT_COLUMNS].sum(axis=1), 1.0)
                and (path[WEIGHT_COLUMNS] >= -1e-10).all().all()
                and (path[WEIGHT_COLUMNS] <= 1.0 + 1e-10).all().all()
                for path in paths.values()
            )
        ),
        "no_directional_gvz_ovx_mu": bool(
            all(
                path["gvz_mu_adjustment_GLD"].eq(0.0).all()
                and path["ovx_mu_adjustment_USO"].eq(0.0).all()
                for path in paths.values()
            )
        ),
        "all_solvers_feasible": bool(
            all(
                path["solver_success"].all()
                and not path["used_fallback"].any()
                and path["volatility_slack"].min() >= -1e-7
                and path["cdar_slack"].min() >= -1e-7
                for path in paths.values()
            )
        ),
    }
    report = {
        "performance": performance,
        "bootstrap": bootstrap,
        "risk_tests": risk_tests,
        "paths": paths,
        "asset_vol_signals": asset_vol,
        "daily_asset_vol": daily_asset_vol,
        "source_checks": source_checks,
        "solver_audit": {
            name: solver_summary(path) for name, path in paths.items()
        },
        "data_audit": {
            "asset_vol": asset_vol_audit,
            "fundamental": fundamental_audit,
            "market": market_audit,
        },
    }
    if save:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        performance.to_csv(
            OUTPUT_DIR / "performance_comparison.csv", index=False
        )
        bootstrap.to_csv(
            OUTPUT_DIR / "paired_block_bootstrap_vs_stage35.csv", index=False
        )
        risk_tests.to_csv(
            OUTPUT_DIR / "asset_risk_predictive_regressions.csv", index=False
        )
        asset_vol.to_csv(
            OUTPUT_DIR / "monthly_asset_volatility_signals.csv"
        )
        daily_asset_vol.to_csv(OUTPUT_DIR / "normalized_gvz_ovx_daily.csv")
        for name, path in paths.items():
            path.to_csv(OUTPUT_DIR / f"{name.lower()}_monthly.csv")
        serializable = {
            "source_checks": source_checks,
            "solver_audit": report["solver_audit"],
            "data_audit": report["data_audit"],
        }
        (OUTPUT_DIR / "colab_validation_report.json").write_text(
            json.dumps(serializable, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
    return report


## 16. 결과 표시와 다운로드 묶음

성과표, 로그 NAV, 월말 drawdown, 인과성 및 solver 검사를 표시합니다. 마지막 함수는 모든 CSV·JSON 결과를 하나의 ZIP으로 묶습니다. Colab에서는 선택적으로 이 결과 ZIP을 바로 내려받을 수 있습니다.

### 결과를 읽을 때 주의할 점

전체 2007~2026 구간에서 Stage36은 Stage35보다 Sharpe와 MDD가 좋아졌지만 CAGR은 약간 낮습니다. 2018 이후에는 CAGR과 Sharpe 모두 Stage35보다 낮으므로 “모든 시기에 우월한 알파”가 아니라 **일부 기대수익을 포기하고 금·원유 위험예산을 더 보수적으로 계산한 전략**으로 읽어야 합니다.

결과표에서 함께 볼 열은 `expected_mu_*`, 자산별 variance multiplier, 실제 `w_*`, `turnover`, `trade_cost`, `solver_success`, 두 위험제약의 slack입니다. 성과 숫자만 보지 않고 어떤 위험센서가 어떤 비중변화를 만들었는지 월별 CSV로 역추적할 수 있습니다.

In [ ]:
def display_stage36_results(report: dict[str, Any]) -> None:
    performance = report["performance"]
    columns = [
        "Strategy",
        "Period",
        "CAGR",
        "Volatility",
        "Sharpe",
        "MDD",
        "AvgTurnover",
    ]
    display_table = performance[columns].copy()
    for column in ["CAGR", "Volatility", "MDD", "AvgTurnover"]:
        display_table[column] = display_table[column].map(
            lambda value: f"{value * 100:.3f}%"
        )
    print("\n성과 비교")
    try:
        from IPython.display import display

        display(display_table)
    except ImportError:
        print(display_table.to_string(index=False))

    paths = report["paths"]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
    colors = {
        "Stage35_Frozen": "#62788d",
        "Stage36_GVZGoldRisk": "#b38a32",
        "Stage36_OVXOilRisk": "#b35b45",
        "Stage36_GVZ_OVXAssetRisk": "#177b6d",
    }
    for name, path in paths.items():
        nav = (1.0 + path["return"]).cumprod()
        axes[0].plot(
            nav.index.to_timestamp(),
            nav,
            label=name,
            color=colors[name],
        )
        axes[1].plot(
            path.index.to_timestamp(),
            path["drawdown"] * 100.0,
            label=name,
            color=colors[name],
        )
    axes[0].set_yscale("log")
    axes[0].set_title("Net NAV (log scale)")
    axes[1].set_title("Monthly drawdown (%)")
    for axis in axes:
        axis.grid(alpha=0.25)
        axis.legend(fontsize=7)
    plt.tight_layout()
    plt.show()

    print("\n인과성·제약 검사")
    print(json.dumps(report["source_checks"], ensure_ascii=False, indent=2))
    print("\nSLSQP 검사")
    print(json.dumps(report["solver_audit"], ensure_ascii=False, indent=2))


def zip_colab_outputs() -> Path:
    assert OUTPUT_DIR is not None
    archive = OUTPUT_DIR.parent / "stage36_colab_outputs.zip"
    if archive.exists():
        archive.unlink()
    with zipfile.ZipFile(
        archive, "w", compression=zipfile.ZIP_DEFLATED
    ) as handle:
        for path in sorted(OUTPUT_DIR.rglob("*")):
            if path.is_file():
                handle.write(path, path.relative_to(OUTPUT_DIR.parent))
    return archive


## 17. 전체 실행

기본값은 원 프로젝트와 같은 12개월 블록·2,000회 bootstrap입니다. 로컬 자동검증에서만 환경변수로 반복수를 낮출 수 있으며 Colab에서는 자동으로 2,000회가 적용됩니다.


In [ ]:
bootstrap_replications = int(os.environ.get("STAGE36_BOOTSTRAP_REPS", "2000"))
report = run_stage36_research(
    save=True,
    bootstrap_replications=bootstrap_replications,
)
display_stage36_results(report)


## 18. 기준 수치와 재현성 감사

Colab·SciPy 버전에 따른 SLSQP의 마지막 자리 차이는 허용하되, 핵심 성과는 기존 Stage36 결과와 0.01%p/0.001 이내여야 합니다. 모든 인과성·제약 검사도 참이어야 합니다.


In [ ]:
expected = {
    ("Stage35_Frozen", "full_2007_2026"): {
        "CAGR": 0.1060756884, "Sharpe": 1.0573641671, "MDD": -0.1374348684,
    },
    ("Stage36_GVZ_OVXAssetRisk", "full_2007_2026"): {
        "CAGR": 0.1049938875, "Sharpe": 1.1049112901, "MDD": -0.1240708722,
    },
}
indexed = report["performance"].set_index(["Strategy", "Period"])
for key, targets in expected.items():
    for metric, target in targets.items():
        actual = float(indexed.loc[key, metric])
        tolerance = 1e-4 if metric != "Sharpe" else 1e-3
        assert abs(actual - target) <= tolerance, (key, metric, actual, target)
assert all(report["source_checks"].values()), report["source_checks"]
print("✅ Stage35·Stage36 핵심 성과와 모든 인과성·제약 검사가 통과했습니다.")


## 19. 주요 결과 파일 묶기

다음 셀은 월별 비중, 성과표, 위험회귀, bootstrap, validation JSON을 `stage36_colab_outputs.zip`으로 묶습니다. Colab이면 다운로드를 시작하고, 로컬이면 생성 경로만 출력합니다.


In [ ]:
output_zip = zip_colab_outputs()
print(f"결과 ZIP: {output_zip}")
try:
    from google.colab import files
    files.download(str(output_zip))
except ImportError:
    pass
